In [2]:
import pandas as pd
from IPython.display import display

# Если данные ещё не в памяти:
edges = pd.read_parquet("edges.parquet")
nodes = pd.read_parquet("nodes.parquet")
transactions = pd.read_parquet("transactions.parquet")

dfs = {"edges": edges, "nodes": nodes, "transactions": transactions}

# показывать все строки и колонки без обрезки
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def summary(df):
    return pd.DataFrame({
        "dtype":     df.dtypes.astype(str),
        "non_null":  df.notna().sum(),
        "nulls":     df.isna().sum(),
        "null_%":    (df.isna().mean() * 100).round(2),
        "n_unique":  df.nunique(dropna=False),
        "example":   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

for name, df in dfs.items():
    print(f"\n{'='*80}\n{name}: {df.shape[0]:,} строк × {df.shape[1]} колонок\n{'='*80}")
    display(summary(df))
    print("Дубликатов строк:", df.duplicated().sum())


edges: 3,119 строк × 5 колонок


,dtype,non_null,nulls,null_%,n_unique,example
src,int64,3119,0,0.00,694,"100,000,003,684,369,104.00"
dst,int64,3119,0,0.00,2206,"100,000,003,037,660,096.00"
sum_kzt,float64,3119,0,0.00,1484,"53,000.00"
n_tx,int64,3119,0,0.00,22,1.00
depth,int8,3119,0,0.00,4,1.00


Дубликатов строк: 0

nodes: 2,248 строк × 3 колонок


,dtype,non_null,nulls,null_%,n_unique,example
gid,int64,2248,0,0.00,2248,100000000343175100
depth,int64,2248,0,0.00,5,0
is_seed,bool,2248,0,0.00,2,True


Дубликатов строк: 0

transactions: 4,840 строк × 4 колонок


,dtype,non_null,nulls,null_%,n_unique,example
src,int64,4840,0,0.00,694,100000002175422100
dst,int64,4840,0,0.00,2206,100000003004487100
date,object,4840,0,0.00,31,2026-07-01
sum_kzt,float64,4840,0,0.00,1583,"50,000.00"


Дубликатов строк: 97


In [3]:
import numpy as np, pandas as pd, networkx as nx
from IPython.display import display, IFrame

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

nodes_ = nodes.copy()
e = edges.copy()
tx = transactions.copy()
tx["date"] = pd.to_datetime(tx["date"])
nodes_["is_seed"] = nodes_["is_seed"].astype(bool)

seeds = set(nodes_.loc[nodes_.is_seed, "gid"])
e["from_seed"] = e["src"].isin(seeds)

print("Дубли пар src→dst в edges:", e.duplicated(["src", "dst"]).sum())
print("Петли src==dst:", (e.src == e.dst).sum())

Дубли пар src→dst в edges: 0
Петли src==dst: 0


In [4]:
out_ = e.groupby("src").agg(out_sum=("sum_kzt", "sum"), out_tx=("n_tx", "sum"), out_deg=("dst", "nunique"))
in_  = e.groupby("dst").agg(in_sum=("sum_kzt", "sum"),  in_tx=("n_tx", "sum"),  in_deg=("src", "nunique"))
in_seed = (e[e.from_seed].groupby("dst")
           .agg(in_from_seed_sum=("sum_kzt", "sum"), in_from_seed_deg=("src", "nunique")))

# активность по датам (как отправитель или получатель)
act = pd.concat([tx[["src", "date"]].rename(columns={"src": "gid"}),
                 tx[["dst", "date"]].rename(columns={"dst": "gid"})])
act = act.groupby("gid")["date"].agg(first_date="min", last_date="max", active_days="nunique")

reg = nodes_.set_index("gid").join([out_, in_, in_seed, act])
num_cols = ["out_sum", "out_tx", "out_deg", "in_sum", "in_tx", "in_deg", "in_from_seed_sum", "in_from_seed_deg"]
reg[num_cols] = reg[num_cols].fillna(0)

# уникальные контрагенты (и входящие, и исходящие)
pairs = pd.concat([e[["src", "dst"]].rename(columns={"src": "gid", "dst": "cp"}),
                   e[["dst", "src"]].rename(columns={"dst": "gid", "src": "cp"})])
reg["n_counterparties"] = pairs.groupby("gid")["cp"].nunique()
reg["n_counterparties"] = reg["n_counterparties"].fillna(0).astype(int)

# производные метрики
reg["total_volume"] = reg.in_sum + reg.out_sum            # оборот: вход + выход
reg["throughput"]   = np.minimum(reg.in_sum, reg.out_sum) # сколько «прошло насквозь»
reg["net_flow"]     = reg.in_sum - reg.out_sum            # >0 осело, <0 отдал больше, чем видно на входе
reg["pass_ratio"]   = np.where(reg.in_sum > 0, reg.out_sum / reg.in_sum, np.nan)

# флаги артефактов данных
reg["flag_truncated"]   = (reg.depth == 4) & (reg.out_deg == 0)   # обрыв обхода, а не «сток»
reg["flag_seed_no_out"] = reg.is_seed & (reg.out_deg == 0)
reg["flag_out_gt_in"]   = (reg.out_sum > reg.in_sum) & ~reg.is_seed
reg["flag_isolated"]    = (reg.in_deg + reg.out_deg) == 0

print(f"Узлов: {len(reg):,}")
print("Обрыв 4-го колена:", reg.flag_truncated.sum(), "(в ТЗ 444)")
print("Seed без исходящих:", reg.flag_seed_no_out.sum(), "(в ТЗ 31)")
print("Отдают больше, чем получили (не seed):", reg.flag_out_gt_in.sum())
print("Изолированные узлы:", reg.flag_isolated.sum())
display(reg.sort_values("total_volume", ascending=False).head(10))

Узлов: 2,248
Обрыв 4-го колена: 444 (в ТЗ 444)
Seed без исходящих: 31 (в ТЗ 31)
Отдают больше, чем получили (не seed): 335
Изолированные узлы: 19


,depth,is_seed,out_sum,out_tx,out_deg,in_sum,in_tx,in_deg,in_from_seed_sum,in_from_seed_deg,first_date,last_date,active_days,n_counterparties,total_volume,throughput,net_flow,pass_ratio,flag_truncated,flag_seed_no_out,flag_out_gt_in,flag_isolated
gid,,,,,,,,,,,,,,,,,,,,,,
100000000331309100,2,False,"23,001,375",126,99,"984,635",8,5,0,0,2026-07-01,2026-07-31,29,104,"23,986,010","984,635","-22,016,740",23,False,False,True,False
100000002224132100,2,False,"12,115,000",25,4,"3,951,020",10,5,0,0,2026-07-02,2026-07-29,15,6,"16,066,020","3,951,020","-8,163,980",3,False,False,True,False
100000003684369100,0,True,"8,588,655",67,62,"3,848,436",58,24,"233,000",1,2026-07-12,2026-07-28,16,85,"12,437,091","3,848,436","-4,740,219",2,False,False,False,False
100000003016635100,0,True,"9,414,081",151,73,"586,981",36,8,0,0,2026-07-03,2026-07-23,21,75,"10,001,062","586,981","-8,827,100",16,False,False,False,False
100000005242320100,2,False,"8,936,295",29,21,"365,700",2,2,0,0,2026-07-01,2026-07-21,16,23,"9,301,995","365,700","-8,570,595",24,False,False,True,False
100000000437046100,2,False,"7,606,224",73,42,"1,107,200",20,9,0,0,2026-07-10,2026-07-31,21,48,"8,713,424","1,107,200","-6,499,024",7,False,False,True,False
100000008603629100,2,False,"4,986,156",74,61,"1,817,300",45,19,0,0,2026-07-03,2026-07-31,27,73,"6,803,456","1,817,300","-3,168,856",3,False,False,True,False
100000004400305100,2,False,"5,830,181",101,82,"222,800",9,7,0,0,2026-07-03,2026-07-29,22,87,"6,052,981","222,800","-5,607,381",26,False,False,True,False
100000002963189100,3,False,"5,219,000",15,7,"490,000",3,1,0,0,2026-07-08,2026-07-25,7,7,"5,709,000","490,000","-4,729,000",11,False,False,True,False


In [5]:
tx_agg = tx.groupby(["src", "dst"]).agg(
    tx_sum=("sum_kzt", "sum"), tx_n=("sum_kzt", "size"),
    min_tx=("sum_kzt", "min"), max_tx=("sum_kzt", "max"),
    first_date=("date", "min"), last_date=("date", "max"), active_days=("date", "nunique"))

edge_reg = (e.merge(tx_agg, on=["src", "dst"], how="left")
             .merge(reg[["depth", "is_seed"]].add_prefix("src_"), left_on="src", right_index=True)
             .merge(reg[["depth", "is_seed"]].add_prefix("dst_"), left_on="dst", right_index=True))

pair_set = set(zip(e.src, e.dst))
edge_reg["mutual"] = [(d, s) in pair_set for s, d in zip(edge_reg.src, edge_reg.dst)]  # A→B и B→A
edge_reg["back_edge"] = edge_reg.dst_depth <= edge_reg.src_depth                        # деньги идут «назад/вбок»
edge_reg["sum_mismatch"] = (edge_reg.sum_kzt - edge_reg.tx_sum).abs() > 1
edge_reg = edge_reg.sort_values("sum_kzt", ascending=False)

print("Расхождений edges vs transactions:", edge_reg.sum_mismatch.sum())
print("Взаимных пар:", edge_reg.mutual.sum(), "| обратных/боковых рёбер:", edge_reg.back_edge.sum())
display(edge_reg.head(20))

# матрица потоков между коленами: откуда и куда идут деньги
print("\nОборот между коленами (строки: колено отправителя, колонки: колено получателя):")
display(pd.pivot_table(edge_reg, index="src_depth", columns="dst_depth",
                       values="sum_kzt", aggfunc="sum", fill_value=0))

# граф
G = nx.from_pandas_edgelist(e, "src", "dst", edge_attr=["sum_kzt", "n_tx", "depth"], create_using=nx.DiGraph)
G.add_nodes_from(reg.index)

comps = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
reg["component"] = pd.Series({n: i for i, c in enumerate(comps) for n in c})
reg["component_size"] = reg.groupby("component")["depth"].transform("size")
print(f"\nКомпонент: {len(comps)} | размеры топ-5: {[len(c) for c in comps[:5]]}")
display(reg.groupby("component").agg(n=("depth", "size"), seeds=("is_seed", "sum"),
                                     volume=("out_sum", "sum")).head(16))

Расхождений edges vs transactions: 0
Взаимных пар: 354 | обратных/боковых рёбер: 745


,src,dst,sum_kzt,n_tx,depth,from_seed,tx_sum,tx_n,min_tx,max_tx,first_date,last_date,active_days,src_depth,src_is_seed,dst_depth,dst_is_seed,mutual,back_edge,sum_mismatch
2090,100000002224132100,100000004892144100,"4,400,000",6,3,False,"4,400,000",6,"200,000","1,000,000",2026-07-02,2026-07-16,4,2,False,3,False,True,False,False
1537,100000007594394100,100000006274203100,"3,376,312",40,3,False,"3,376,312",40,"20,000","250,000",2026-07-11,2026-07-31,19,2,False,3,False,True,False,False
1132,100000007442518100,100000008258083100,"3,000,000",1,2,False,"3,000,000",1,"3,000,000","3,000,000",2026-07-21,2026-07-21,1,1,False,2,False,False,False,False
561,100000003835149100,100000002224132100,"2,900,720",6,2,False,"2,900,720",6,"136,450","1,155,800",2026-07-08,2026-07-29,6,1,False,2,False,False,False,False
862,100000000274067100,100000002958343100,"2,900,000",4,2,False,"2,900,000",4,"100,000","2,000,000",2026-07-01,2026-07-13,4,1,False,2,False,True,False,False
2103,100000002224132100,100000000594783100,"2,635,000",10,3,False,"2,635,000",10,"5,000","1,000,000",2026-07-02,2026-07-27,7,2,False,3,False,True,False,False
1330,100000002224132100,100000004603109100,"2,580,000",6,3,False,"2,580,000",6,"180,000","1,000,000",2026-07-02,2026-07-08,5,2,False,3,False,True,False,False
2034,100000002224132100,100000000552584100,"2,500,000",3,3,False,"2,500,000",3,"500,000","1,000,000",2026-07-08,2026-07-16,2,2,False,3,False,False,False,False
186,100000008637537100,100000001732159100,"2,283,100",28,1,True,"2,283,100",28,"10,000","318,000",2026-07-01,2026-07-17,10,0,True,0,True,True,True,False
2441,100000004722765100,100000005075949100,"2,226,000",7,4,False,"2,226,000",7,"10,000","984,000",2026-07-09,2026-07-23,4,3,False,4,False,False,False,False



Оборот между коленами (строки: колено отправителя, колонки: колено получателя):


dst_depth,0,1,2,3,4
src_depth,,,,,
0,"6,609,582","48,684,596",0,0,0
1,"5,506,206","6,214,120","54,064,002",0,0
2,"1,318,644","24,926,442","10,556,837","122,189,653",0
3,"1,414,735","3,031,847","16,644,483","8,056,700","56,672,165"



Компонент: 35 | размеры топ-5: [1877, 270, 17, 13, 6]


,n,seeds,volume
component,,,
0,1877,46,"342,125,465"
1,270,1,"10,178,044"
2,17,1,"2,977,887"
3,13,1,"1,392,916"
4,6,1,"275,885"
5,6,1,"1,615,030"
6,6,1,"383,300"
7,6,1,"1,527,967"
8,5,2,"108,007"


In [6]:
top100 = reg.sort_values("total_volume", ascending=False).head(100).reset_index()
top100.insert(0, "rank", range(1, 101))
cols = ["rank", "gid", "depth", "is_seed", "total_volume", "in_sum", "out_sum", "net_flow", "pass_ratio",
        "in_deg", "out_deg", "n_counterparties", "in_from_seed_sum", "active_days",
        "component", "flag_truncated", "flag_out_gt_in"]
display(top100[cols])

graph_turnover = e.sum_kzt.sum()
print(f"Оборот графа: {graph_turnover:,.0f} KZT (в ТЗ 365 890 012)")
print(f"Доля топ-100 в сумме in+out: {top100.total_volume.sum() / (2 * graph_turnover):.1%}")

# чем топ-100 отличается от сети в целом
print("\nРаспределение по коленам: топ-100 vs вся сеть (%)")
display(pd.DataFrame({"top100_%": top100.depth.value_counts(normalize=True).sort_index() * 100,
                      "all_%":    reg.depth.value_counts(normalize=True).sort_index() * 100}))
print("В топ-100: seed", top100.is_seed.sum(), "| обрыв 4-го колена", top100.flag_truncated.sum(),
      "| out>in", top100.flag_out_gt_in.sum())

# насколько топ зависит от выбранной метрики
print("\nПересечение топ-100 по обороту с топ-100 по другим метрикам:")
top_set = set(top100.gid)
for m in ["in_sum", "out_sum", "throughput", "n_counterparties", "in_deg", "out_deg", "in_from_seed_sum"]:
    print(f"  {m:18s}: {len(top_set & set(reg.nlargest(100, m).index))}")

,rank,gid,depth,is_seed,total_volume,in_sum,out_sum,net_flow,pass_ratio,in_deg,out_deg,n_counterparties,in_from_seed_sum,active_days,component,flag_truncated,flag_out_gt_in
0,1,100000000331309100,2,False,"23,986,010","984,635","23,001,375","-22,016,740",23,5,99,104,0,29,0,False,True
1,2,100000002224132100,2,False,"16,066,020","3,951,020","12,115,000","-8,163,980",3,5,4,6,0,15,0,False,True
2,3,100000003684369100,0,True,"12,437,091","3,848,436","8,588,655","-4,740,219",2,24,62,85,"233,000",16,0,False,False
3,4,100000003016635100,0,True,"10,001,062","586,981","9,414,081","-8,827,100",16,8,73,75,0,21,0,False,False
4,5,100000005242320100,2,False,"9,301,995","365,700","8,936,295","-8,570,595",24,2,21,23,0,16,0,False,True
5,6,100000000437046100,2,False,"8,713,424","1,107,200","7,606,224","-6,499,024",7,9,42,48,0,21,0,False,True
6,7,100000008603629100,2,False,"6,803,456","1,817,300","4,986,156","-3,168,856",3,19,61,73,0,27,0,False,True
7,8,100000004400305100,2,False,"6,052,981","222,800","5,830,181","-5,607,381",26,7,82,87,0,22,0,False,True
8,9,100000002963189100,3,False,"5,709,000","490,000","5,219,000","-4,729,000",11,1,7,7,0,7,0,False,True
9,10,100000008710791100,2,False,"5,703,021","181,510","5,521,511","-5,340,001",30,3,23,26,0,10,0,False,True


Оборот графа: 365,890,012 KZT (в ТЗ 365 890 012)
Доля топ-100 в сумме in+out: 47.5%

Распределение по коленам: топ-100 vs вся сеть (%)


,top100_%,all_%
depth,,
0,11,4
1,23,21
2,34,21
3,27,35
4,5,20


В топ-100: seed 11 | обрыв 4-го колена 5 | out>in 58

Пересечение топ-100 по обороту с топ-100 по другим метрикам:
  in_sum            : 44
  out_sum           : 72
  throughput        : 54
  n_counterparties  : 47
  in_deg            : 32
  out_deg           : 49
  in_from_seed_sum  : 17


In [11]:
from pyvis.network import Network

DEPTH_COLORS = {0: "#d62728", 1: "#ff7f0e", 2: "#2ca02c", 3: "#1f77b4", 4: "#9467bd"}
fmt = lambda x: f"{x:,.0f}".replace(",", " ")
VMAX = np.log1p(reg.total_volume.max())
EMAX = np.log1p(e.sum_kzt.max())

def draw(G_sub, path, highlight=(), label_nodes=None):
    label_nodes = (set(label_nodes) if label_nodes is not None else set()) | set(highlight)
    net = Network(height="850px", width="100%", directed=True, select_menu=True, cdn_resources="in_line")
    for n in G_sub.nodes():
        r = reg.loc[n]
        title = (f"gid {n}{'  [SEED]' if r.is_seed else ''}\nколено: {int(r.depth)}\n"
                 f"получил: {fmt(r.in_sum)} KZT от {int(r.in_deg)}\n"
                 f"отправил: {fmt(r.out_sum)} KZT на {int(r.out_deg)}\n"
                 f"оборот: {fmt(r.total_volume)} KZT"
                 + ("\n⚠ обрыв 4-го колена (исходящие не выгружены)" if r.flag_truncated else ""))
        net.add_node(int(n), label=str(n) if (r.is_seed or n in label_nodes) else " ",
                     title=title, color=DEPTH_COLORS.get(int(r.depth), "#999"),
                     size=float(5 + 30 * np.log1p(r.total_volume) / VMAX),
                     shape="star" if r.is_seed else "dot",
                     borderWidth=5 if n in highlight else 1)
    for u, v, d in G_sub.edges(data=True):
        net.add_edge(int(u), int(v), width=float(0.5 + 6 * np.log1p(d["sum_kzt"]) / EMAX),
                     title=f"{u} → {v}\n{fmt(d['sum_kzt'])} KZT, {int(d['n_tx'])} тр.", color="#999")
    net.set_options("""{
      "physics": {"solver": "forceAtlas2Based",
                  "forceAtlas2Based": {"gravitationalConstant": -40, "springLength": 80},
                  "stabilization": {"iterations": 400}},
      "edges": {"smooth": false, "arrows": {"to": {"enabled": true, "scaleFactor": 0.4}}},
      "interaction": {"hover": true}
    }""")
    # pyvis на Windows пишет в cp1251 — сохраняем сами в UTF-8
    with open(path, "w", encoding="utf-8") as f:
        f.write(net.generate_html())
    return path

# 1) вся сеть: цвет = колено, звезда = seed, размер = оборот, толщина = сумма
draw(G, "graph_full.html", label_nodes=top100.gid)
display(IFrame("graph_full.html", width="100%", height=870))

In [12]:
# 2) топ-100 + все seed и связи между ними (легче читать)
sub_nodes = set(top100.gid) | seeds
draw(G.subgraph(sub_nodes), "graph_top100.html", highlight=set(top100.gid[:20]), label_nodes=sub_nodes)
display(IFrame("graph_top100.html", width="100%", height=870))

In [13]:
findings = []
def note(point, what, value, conclusion=""):
    findings.append({"пункт": point, "проверка": what, "значение": value, "вывод": conclusion})
    print(f"[{point}] {what}: {value}" + (f"\n     → {conclusion}" if conclusion else ""))

# чтобы маленькие доли не округлялись до 0
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.4g}")
TOTAL = e.sum_kzt.sum()

In [14]:
transit = reg.pass_ratio.between(0.8, 1.2) & (reg.in_sum > 0) & (reg.out_sum > 0)
reg["volume_dedup"] = np.maximum(reg.in_sum, reg.out_sum)   # сколько денег прошло через узел, без двойного счёта

top_tv = set(reg.nlargest(100, "total_volume").index)
top_dd = set(reg.nlargest(100, "volume_dedup").index)
t_top, t_all = transit[list(top_tv)].mean(), transit.mean()

note(1, "доля транзитных (pass 0.8–1.2): топ-100 по in+out vs вся сеть", f"{t_top:.0%} vs {t_all:.0%}",
     "транзит перепредставлен в топе" if t_top > 2 * t_all else "сильного перекоса нет")
note(1, "пересечение топ-100 по in+out и по max(in,out)", len(top_tv & top_dd),
     "ранжирование заметно меняется" if len(top_tv & top_dd) < 85 else "почти не меняется")

cmp = pd.DataFrame({"rank_in_out": reg.total_volume.rank(ascending=False),
                    "rank_dedup":  reg.volume_dedup.rank(ascending=False),
                    "pass_ratio":  reg.pass_ratio})
cmp["shift"] = cmp.rank_in_out - cmp.rank_dedup
print("\nСильнее всего опускаются при дедупликации:")
display(cmp.loc[list(top_tv)].sort_values("shift").head(10))

[1] доля транзитных (pass 0.8–1.2): топ-100 по in+out vs вся сеть: 2% vs 3%
     → сильного перекоса нет
[1] пересечение топ-100 по in+out и по max(in,out): 91
     → почти не меняется

Сильнее всего опускаются при дедупликации:


,rank_in_out,rank_dedup,pass_ratio,shift
gid,,,,
100000008165763100,53,111,1.088,-58
100000004135268100,58,115,0.8073,-57
100000007629096100,82,132,0.7117,-50
100000004221668100,79,123,0.6483,-44
100000002849158100,65,106,1.816,-40.5
100000001887715100,62,102,0.5633,-40
100000000791285100,89,125,0.5804,-36
100000006889963100,76,95,0.3099,-19
100000008512552100,91,109,3.302,-18


In [15]:
ng = reg[~reg.is_seed]
gap = ng[ng.flag_out_gt_in].assign(gap=lambda d: d.out_sum - d.in_sum)

note(2, "не-seed узлов с out > in", len(gap), "у них не виден часть входа, баланс считать нельзя")
note(2, "невидимый вход у них в сумме, KZT", fmt(gap.gap.sum()), f"= {gap.gap.sum() / TOTAL:.1%} оборота графа")

first_out = tx.groupby("src").date.min().reindex(gap.index)
first_in  = tx.groupby("dst").date.min().reindex(gap.index)
n_early = int((first_out < first_in).sum())
note(2, "из них первый исходящий раньше первого видимого входящего", n_early,
     "деньги были на счёте до выборки, т.е. вход из-за пределов графа")

print("\nПо коленам:"); display(gap.depth.value_counts().sort_index())
display(gap.sort_values("gap", ascending=False).head(15)
        [["depth", "in_sum", "out_sum", "gap", "in_deg", "out_deg"]])

[2] не-seed узлов с out > in: 335
     → у них не виден часть входа, баланс считать нельзя
[2] невидимый вход у них в сумме, KZT: 229 908 004
     → = 62.8% оборота графа
[2] из них первый исходящий раньше первого видимого входящего: 186
     → деньги были на счёте до выборки, т.е. вход из-за пределов графа

По коленам:


depth
1     80
2    110
3    145
Name: count, dtype: int64

,depth,in_sum,out_sum,gap,in_deg,out_deg
gid,,,,,,
100000000331309100,2,"984,635","23,001,375","22,016,740",5,99
100000005242320100,2,"365,700","8,936,295","8,570,595",2,21
100000002224132100,2,"3,951,020","12,115,000","8,163,980",5,4
100000000437046100,2,"1,107,200","7,606,224","6,499,024",9,42
100000004400305100,2,"222,800","5,830,181","5,607,381",7,82
100000008710791100,2,"181,510","5,521,511","5,340,001",3,23
100000008547948100,2,"212,665","5,185,600","4,972,935",6,26
100000002963189100,3,"490,000","5,219,000","4,729,000",1,7
100000003835149100,1,"248,500","4,600,440","4,351,940",1,2


In [16]:
no_out    = reg.out_deg == 0
trunc     = reg.flag_truncated
real_sink = no_out & (reg.depth < 4) & ~reg.is_seed     # у колен 0–3 исходящие выгружены, значит сток видимый
reg["flag_real_sink"] = real_sink

note(3, "узлов без исходящих всего", int(no_out.sum()))
note(3, "из них обрыв 4-го колена", int(trunc.sum()), "не terminal, а «нет данных»")
note(3, "кандидаты в настоящие стоки (колено<4, не seed, out=0)", int(real_sink.sum()),
     f"сумма входа {fmt(reg.loc[real_sink, 'in_sum'].sum())} KZT")
note(3, "доля денег графа, ушедших в обрезанные узлы", f"{reg.loc[trunc, 'in_sum'].sum() / TOTAL:.1%}")
note(3, "обрезанных в топ-100 по входу", int(trunc[reg.nlargest(100, 'in_sum').index].sum()),
     "наивное правило «много пришло и не ушло» поймает их")

[3] узлов без исходящих всего: 1554
[3] из них обрыв 4-го колена: 444
     → не terminal, а «нет данных»
[3] кандидаты в настоящие стоки (колено<4, не seed, out=0): 1079
     → сумма входа 136 739 480 KZT
[3] доля денег графа, ушедших в обрезанные узлы: 15.5%
[3] обрезанных в топ-100 по входу: 17
     → наивное правило «много пришло и не ушло» поймает их


In [17]:
idx = {g: i for i, g in enumerate(reg.index)}
s = e.src.map(idx).values; d = e.dst.map(idx).values; w = e.sum_kzt.values
denom = np.maximum(reg.in_sum.values, reg.out_sum.values)
seed_arr = reg.is_seed.values

share = seed_arr.astype(float)
for it in range(300):
    contrib = np.bincount(d, weights=w * share[s], minlength=len(reg))
    new = np.where(denom > 0, np.minimum(contrib / np.where(denom > 0, denom, 1), 1), 0)
    new[seed_arr] = 1.0
    if np.abs(new - share).max() < 1e-10:
        share = new; break
    share = new
contrib = np.bincount(d, weights=w * share[s], minlength=len(reg))
reg["seed_share"] = share             # какая доля денег узла — предположительно от seed
reg["seed_money_in"] = contrib        # сколько «seed-денег» пришло, KZT
print("Итераций до сходимости:", it + 1)

top_seed = set(reg.nlargest(100, "seed_money_in").index)
low = reg.loc[list(top_tv)].query("seed_share < 0.1 and not is_seed")
note(4, "пересечение топ-100 по обороту и по seed-деньгам", len(top_tv & top_seed),
     "оборот плохо заменяет связь с seed" if len(top_tv & top_seed) < 70 else "метрики близки")
note(4, "узлов в топ-100 по обороту, где seed-денег < 10%", len(low),
     "кандидаты в «посторонние» крупные счета")
note(4, "Spearman: оборот vs seed-деньги",
     round(reg[["total_volume", "seed_money_in"]].corr("spearman").iloc[0, 1], 2))

print("\nДоля seed-денег по коленам:")
display(reg.groupby("depth").seed_share.describe()[["mean", "25%", "50%", "75%"]])
display(low.sort_values("total_volume", ascending=False).head(10)
        [["depth", "total_volume", "in_sum", "seed_money_in", "seed_share", "in_deg"]])

Итераций до сходимости: 121
[4] пересечение топ-100 по обороту и по seed-деньгам: 20
     → оборот плохо заменяет связь с seed
[4] узлов в топ-100 по обороту, где seed-денег < 10%: 77
     → кандидаты в «посторонние» крупные счета
[4] Spearman: оборот vs seed-деньги: 0.39

Доля seed-денег по коленам:


,mean,25%,50%,75%
depth,,,,
0,1,1,1,1
1,0.7697,0.5349,1,1
2,0.2099,0.01719,0.07957,0.2493
3,0.02293,0.0006645,0.001845,0.007631
4,0.008658,0.0002427,0.001137,0.004209


,depth,total_volume,in_sum,seed_money_in,seed_share,in_deg
gid,,,,,,
100000000331309100,2,"23,986,010","984,635","27,495",0.001195,5
100000002224132100,2,"16,066,020","3,951,020","181,059",0.01495,5
100000005242320100,2,"9,301,995","365,700","2,866",0.0003207,2
100000000437046100,2,"8,713,424","1,107,200","36,890",0.00485,9
100000008603629100,2,"6,803,456","1,817,300","33,165",0.006652,19
100000004400305100,2,"6,052,981","222,800","4,935",0.0008464,7
100000002963189100,3,"5,709,000","490,000",542,0.0001038,1
100000008710791100,2,"5,703,021","181,510","6,961",0.001261,3
100000008547948100,2,"5,398,265","212,665","5,830",0.001124,6


In [18]:
hubs = reg[reg.in_deg >= 8].copy()
hubs["avg_payment"]      = hubs.in_sum / hubs.in_tx
hubs["in_days"]          = tx.groupby("dst").date.apply(lambda x: x.dt.date.nunique())
hubs["amount_cv"]        = tx.groupby("dst").sum_kzt.agg(lambda x: x.std() / x.mean() if len(x) > 1 else np.nan)
payer_outdeg = e.src.map(reg.out_deg)
hubs["payers_exclusive"] = e.assign(p=payer_outdeg == 1).groupby("dst").p.mean()   # доля плательщиков, платящих только ему

merchant_like  = (hubs.seed_share < 0.2) & (hubs.in_days >= 15) & (hubs.pass_ratio.fillna(0) < 0.2)
collector_like = (hubs.seed_share >= 0.5) & (hubs.payers_exclusive >= 0.5)

note(5, "узлов с 8+ плательщиками", len(hubs))
note(5, "похожи на легальный хаб (мало seed-денег, вход ≥15 дней, почти не отдают)", int(merchant_like.sum()),
     "не называть их консолидаторами без оговорки")
note(5, "похожи на сборщика (≥50% seed-денег, плательщики эксклюзивны)", int(collector_like.sum()))

cols5 = ["depth", "in_deg", "in_sum", "seed_share", "in_days", "avg_payment", "amount_cv",
         "payers_exclusive", "pass_ratio", "out_deg"]
print("\nПохожи на хаб:");     display(hubs[merchant_like].sort_values("in_deg", ascending=False)[cols5])
print("\nПохожи на сборщика:"); display(hubs[collector_like].sort_values("in_deg", ascending=False)[cols5])

[5] узлов с 8+ плательщиками: 17
[5] похожи на легальный хаб (мало seed-денег, вход ≥15 дней, почти не отдают): 0
     → не называть их консолидаторами без оговорки
[5] похожи на сборщика (≥50% seed-денег, плательщики эксклюзивны): 0

Похожи на хаб:


,depth,in_deg,in_sum,seed_share,in_days,avg_payment,amount_cv,payers_exclusive,pass_ratio,out_deg
gid,,,,,,,,,,



Похожи на сборщика:


,depth,in_deg,in_sum,seed_share,in_days,avg_payment,amount_cv,payers_exclusive,pass_ratio,out_deg
gid,,,,,,,,,,


In [19]:
reg["pagerank"]    = pd.Series(nx.pagerank(G, weight="sum_kzt"))
reg["betweenness"] = pd.Series(nx.betweenness_centrality(G))   # точный расчёт, несколько секунд

for m in ["pagerank", "betweenness"]:
    t = reg.nlargest(50, m)
    note(6, f"топ-50 по {m}: распределение по коленам",
         t.depth.value_counts().sort_index().to_dict(),
         f"обрезанных узлов: {int(t.flag_truncated.sum())}")

print("\nСредние по коленам:")
display(reg.groupby("depth")[["pagerank", "betweenness"]].mean())

[6] топ-50 по pagerank: распределение по коленам: {0: 7, 1: 12, 2: 21, 3: 10}
     → обрезанных узлов: 0
[6] топ-50 по betweenness: распределение по коленам: {0: 4, 1: 15, 2: 20, 3: 11}
     → обрезанных узлов: 0

Средние по коленам:


,pagerank,betweenness
depth,,
0,0.0005991,0.0002172
1,0.0004266,8.183e-05
2,0.0005506,0.0001372
3,0.0003907,4.089e-05
4,0.0004222,0


In [20]:
sd = reg[reg.is_seed]
in_edges = set(e.src) | set(e.dst)
only_recv = sd[(sd.out_deg == 0) & (sd.in_deg > 0)]

note(7, "seed без исходящих", int((sd.out_deg == 0).sum()), "в ТЗ 31")
note(7, "seed вообще нет в рёбрах", int((~sd.index.isin(list(in_edges))).sum()), "в ТЗ 19; роль = «нет данных»")
note(7, "seed только получатели", len(only_recv), "в ТЗ 12")
note(7, "seed, получающие деньги от других seed", int((sd.in_from_seed_deg > 0).sum()),
     "связи между seed — сигнал общей структуры")

# кто платит seed-получателям: другие seed или узлы глубже (возвратный поток)
payers = edge_reg[edge_reg.dst.isin(only_recv.index)]
print("\nКолено плательщиков для seed-получателей:")
display(payers.src_depth.value_counts().sort_index())
print("\nSeed по компонентам:")
display(sd.groupby("component").size().rename("n_seed").to_frame().join(
        reg.groupby("component").size().rename("component_size")))

[7] seed без исходящих: 31
     → в ТЗ 31
[7] seed вообще нет в рёбрах: 19
     → в ТЗ 19; роль = «нет данных»
[7] seed только получатели: 12
     → в ТЗ 12
[7] seed, получающие деньги от других seed: 18
     → связи между seed — сигнал общей структуры

Колено плательщиков для seed-получателей:


src_depth
0    7
1    5
2    9
3    1
Name: count, dtype: int64


Seed по компонентам:


,n_seed,component_size
component,,
0,46,1877
1,1,270
2,1,17
3,1,13
4,1,6
5,1,6
6,1,6
7,1,6
8,2,5


In [21]:
amt = tx.sum_kzt
note(8, "минимальная сумма транзакции", fmt(amt.min()), "порог соблюдён" if amt.min() >= 5000 else "есть суммы ниже порога!")
note(8, "доля транзакций 5–6 тыс.", f"{amt.between(5000, 5999.99).mean():.1%}",
     "скопление у порога = признак, что ниже есть ещё")
note(8, "доля круглых сумм (кратно 1000)", f"{(amt % 1000 == 0).mean():.1%}")

same_day = tx.groupby(["src", "dst", tx.date.dt.date]).size()
note(8, "пар «отправитель–получатель–день» с 2+ переводами", int((same_day >= 2).sum()),
     "кандидаты в дробление выше порога")

bins = [5000, 6000, 7000, 8000, 9000, 10000, 15000, 20000, 50000, 100000, 500000, np.inf]
display(pd.cut(amt, bins, right=False).value_counts().sort_index().rename("n_tx").to_frame())
display(same_day[same_day >= 2].sort_values(ascending=False).head(15).rename("n_tx_same_day").to_frame())

[8] минимальная сумма транзакции: 5 000
     → порог соблюдён
[8] доля транзакций 5–6 тыс.: 6.9%
     → скопление у порога = признак, что ниже есть ещё
[8] доля круглых сумм (кратно 1000): 65.6%
[8] пар «отправитель–получатель–день» с 2+ переводами: 375
     → кандидаты в дробление выше порога


,n_tx
sum_kzt,
"[5000.0, 6000.0)",335
"[6000.0, 7000.0)",135
"[7000.0, 8000.0)",103
"[8000.0, 9000.0)",88
"[9000.0, 10000.0)",95
"[10000.0, 15000.0)",588
"[15000.0, 20000.0)",366
"[20000.0, 50000.0)",1369
"[50000.0, 100000.0)",705


n_tx_same_day
src                dst                date                     
100000004269433100 100000008418835100 2026-07-12             11
                                      2026-07-16              9
                                      2026-07-14              9
                                      2026-07-09              8
100000000437046100 100000007055802100 2026-07-15              8
100000008628231100 100000005664632100 2026-07-26              8
100000007390016100 100000008324800100 2026-07-07              7
100000008324800100 100000007390016100 2026-07-17              7
100000007170072100 100000003345947100 2026-07-13              6
100000008637537100 100000001732159100 2026-07-04              6
100000004269433100 100000008418835100 2026-07-18              6
100000004388983100 100000005034574100 2026-07-28              6
                                      2026-07-31              6
100000002645993100 100000002398779100 2026-07-07              5
100000008637537100 100000001732159100 2026-07-13              5

In [22]:
note(9, "период транзакций", f"{tx.date.min().date()} — {tx.date.max().date()}")
last_in = tx.groupby("dst").date.max().reindex(reg.index[real_sink])
late = last_in >= tx.date.max() - pd.Timedelta(days=3)
note(9, "настоящих стоков с последним входом в последние 3 дня", int(late.sum()),
     "могли переслать деньги уже в августе, terminal под сомнением")

daily = tx.groupby(tx.date.dt.date).sum_kzt.agg(n_tx="size", sum_kzt="sum")
display(daily)

[9] период транзакций: 2026-07-01 — 2026-07-31
[9] настоящих стоков с последним входом в последние 3 дня: 171
     → могли переслать деньги уже в августе, terminal под сомнением


,n_tx,sum_kzt
date,,
2026-07-01,115,"7,179,461"
2026-07-02,131,"9,362,006"
2026-07-03,181,"12,619,868"
2026-07-04,110,"5,949,727"
2026-07-05,127,"7,709,834"
2026-07-06,124,"5,313,603"
2026-07-07,150,"9,882,395"
2026-07-08,190,"16,175,687"
2026-07-09,156,"10,483,391"


In [23]:
note(10, "edges.depth == колено отправителя", f"{(edge_reg.depth == edge_reg.src_depth).mean():.0%}")
note(10, "edges.depth == колено отправителя + 1", f"{(edge_reg.depth == edge_reg.src_depth + 1).mean():.0%}")
note(10, "edges.depth == колено получателя", f"{(edge_reg.depth == edge_reg.dst_depth).mean():.0%}")
display(pd.crosstab(edge_reg.depth, edge_reg.src_depth, margins=True))

back = edge_reg[edge_reg.back_edge]
note(10, "обратных/боковых рёбер", f"{len(back)} на {fmt(back.sum_kzt.sum())} KZT")
note(10, "из них ведут в seed", int(back.dst_is_seed.sum()), "деньги возвращаются к известным участникам")
note(10, "взаимных пар A↔B", int(edge_reg.mutual.sum() // 2))

from itertools import islice
try:
    cycles = list(nx.simple_cycles(G, length_bound=5))
except TypeError:                      # старый networkx без length_bound
    cycles = [c for c in islice(nx.simple_cycles(G), 20000) if len(c) <= 5]
reg["in_cycle"] = reg.index.isin({n for c in cycles for n in c})
note(10, "циклов длиной ≤5", len(cycles), f"узлов в циклах: {int(reg.in_cycle.sum())}")
display(pd.Series([len(c) for c in cycles]).value_counts().sort_index().rename("n_cycles").to_frame())

[10] edges.depth == колено отправителя: 0%
[10] edges.depth == колено отправителя + 1: 100%
[10] edges.depth == колено получателя: 76%


src_depth,0,1,2,3,All
depth,,,,,
1,520,0,0,0,520
2,0,640,0,0,640
3,0,0,1200,0,1200
4,0,0,0,759,759
All,520,640,1200,759,3119


[10] обратных/боковых рёбер: 745 на 84 279 596 KZT
[10] из них ведут в seed: 125
     → деньги возвращаются к известным участникам
[10] взаимных пар A↔B: 177
[10] циклов длиной ≤5: 468
     → узлов в циклах: 300


,n_cycles
2,177
3,41
4,169
5,81


In [24]:
from sklearn.metrics import adjusted_rand_score

# неориентированный граф с суммой весов в обе стороны
U = nx.Graph(); U.add_nodes_from(G)
for u, v, dd in G.edges(data=True):
    if U.has_edge(u, v): U[u][v]["w"] += dd["sum_kzt"]
    else: U.add_edge(u, v, w=dd["sum_kzt"])
for _, _, dd in U.edges(data=True):
    dd["logw"] = np.log1p(dd["w"])

def labels(comms):
    lab = {n: i for i, c in enumerate(comms) for n in c}
    return np.array([lab[n] for n in reg.index])

runs = {(wt, sd_): labels(nx.community.louvain_communities(U, weight=wt, seed=sd_))
        for wt in ["logw", "w", None] for sd_ in range(5)}

for wt in ["logw", "w", None]:
    aris = [adjusted_rand_score(runs[(wt, 0)], runs[(wt, k)]) for k in range(1, 5)]
    note(11, f"устойчивость Louvain между запусками (вес={wt}), ARI", round(np.mean(aris), 3),
         "стабильно" if np.mean(aris) > 0.9 else "кластеры плавают, фиксировать seed и проверять")
note(11, "совпадение разбиений log-вес vs сырой вес, ARI",
     round(adjusted_rand_score(runs[("logw", 0)], runs[("w", 0)]), 3), "выбор веса влияет на кластеры")

lab0 = runs[("logw", 0)]
cs = (pd.DataFrame({"c": lab0, "seed": reg.is_seed.values, "comp": reg.component.values})
        .groupby("c").agg(n=("seed", "size"), seeds=("seed", "sum"), n_comp=("comp", "nunique")))
note(11, "сообществ всего / ≥5 узлов / с >1 seed",
     f"{len(cs)} / {(cs.n >= 5).sum()} / {(cs.seeds > 1).sum()}", "в ТЗ ориентир 8 сообществ с >1 seed")
note(11, "сообществ внутри крупнейшей компоненты", int(cs[cs.index.isin(np.unique(lab0[reg.component.values == 0]))].shape[0]))
reg["louvain"] = lab0
display(cs.sort_values("n", ascending=False).head(15))

[11] устойчивость Louvain между запусками (вес=logw), ARI: 0.675
     → кластеры плавают, фиксировать seed и проверять
[11] устойчивость Louvain между запусками (вес=w), ARI: 0.899
     → кластеры плавают, фиксировать seed и проверять
[11] устойчивость Louvain между запусками (вес=None), ARI: 0.727
     → кластеры плавают, фиксировать seed и проверять
[11] совпадение разбиений log-вес vs сырой вес, ARI: 0.432
     → выбор веса влияет на кластеры
[11] сообществ всего / ≥5 узлов / с >1 seed: 64 / 39 / 9
     → в ТЗ ориентир 8 сообществ с >1 seed
[11] сообществ внутри крупнейшей компоненты: 28


,n,seeds,n_comp
c,,,
24,203,1,1
3,174,13,1
34,169,1,1
8,135,4,1
27,120,0,1
1,111,1,1
10,108,0,1
0,103,1,1
41,91,0,1


In [25]:
metrics = ["total_volume", "volume_dedup", "in_sum", "out_sum", "throughput", "seed_money_in",
           "in_deg", "out_deg", "n_counterparties", "pagerank", "betweenness"]
tops = {m: set(reg.nlargest(100, m).index) for m in metrics}

print("Пересечения топ-100 (из 100):")
display(pd.DataFrame([[len(tops[a] & tops[b]) for b in metrics] for a in metrics], index=metrics, columns=metrics))
print("Ранговые корреляции Spearman:")
display(reg[metrics].corr("spearman").round(2))

cnt = pd.Series([g for m in metrics for g in tops[m]]).value_counts()
reg["n_top_lists"] = cnt.reindex(reg.index).fillna(0).astype(int)
note(12, "узлов в топ-100 хотя бы по одной метрике", len(cnt))
note(12, "узлов в топ-100 по ≥6 метрикам из 11", int((cnt >= 6).sum()),
     "устойчивое ядро, кандидаты в начало приоритетного списка")
display(reg.loc[cnt[cnt >= 6].index,
        ["depth", "is_seed", "n_top_lists", "total_volume", "seed_money_in", "in_deg", "out_deg",
         "pass_ratio", "flag_truncated"]].sort_values("n_top_lists", ascending=False))

Пересечения топ-100 (из 100):


,total_volume,volume_dedup,in_sum,out_sum,throughput,seed_money_in,in_deg,out_deg,n_counterparties,pagerank,betweenness
total_volume,100,91,44,72,54,20,32,49,47,19,38
volume_dedup,91,100,40,73,45,20,29,52,49,16,39
in_sum,44,40,100,16,36,24,31,11,14,26,12
out_sum,72,73,16,100,48,13,25,62,59,10,45
throughput,54,45,36,48,100,29,36,31,33,27,25
seed_money_in,20,20,24,13,29,100,20,8,10,23,7
in_deg,32,29,31,25,36,20,100,29,41,27,34
out_deg,49,52,11,62,31,8,29,100,88,3,64
n_counterparties,47,49,14,59,33,10,41,88,100,7,61
pagerank,19,16,26,10,27,23,27,3,7,100,7


Ранговые корреляции Spearman:


,total_volume,volume_dedup,in_sum,out_sum,throughput,seed_money_in,in_deg,out_deg,n_counterparties,pagerank,betweenness
total_volume,1,0.99,0.86,0.55,0.53,0.39,0.47,0.52,0.6,0.42,0.49
volume_dedup,0.99,1,0.87,0.48,0.46,0.38,0.44,0.45,0.55,0.41,0.42
in_sum,0.86,0.87,1,0.19,0.28,0.43,0.48,0.18,0.34,0.46,0.21
out_sum,0.55,0.48,0.19,1,0.96,0.24,0.36,0.99,0.81,0.22,0.9
throughput,0.53,0.46,0.28,0.96,1,0.31,0.44,0.95,0.81,0.28,0.91
seed_money_in,0.39,0.38,0.43,0.24,0.31,1,0.38,0.25,0.32,0.39,0.25
in_deg,0.47,0.44,0.48,0.36,0.44,0.38,1,0.37,0.7,0.39,0.44
out_deg,0.52,0.45,0.18,0.99,0.95,0.25,0.37,1,0.83,0.21,0.91
n_counterparties,0.6,0.55,0.34,0.81,0.81,0.32,0.7,0.83,1,0.26,0.84
pagerank,0.42,0.41,0.46,0.22,0.28,0.39,0.39,0.21,0.26,1,0.19


[12] узлов в топ-100 хотя бы по одной метрике: 419
[12] узлов в топ-100 по ≥6 метрикам из 11: 56
     → устойчивое ядро, кандидаты в начало приоритетного списка


,depth,is_seed,n_top_lists,total_volume,seed_money_in,in_deg,out_deg,pass_ratio,flag_truncated
100000003684369100,0,True,11,"12,437,091","500,392",24,62,2.232,False
100000008603629100,2,False,10,"6,803,456","33,165",19,61,2.744,False
100000008686313100,1,False,9,"2,246,516","233,150",6,7,0.1745,False
100000006866783100,0,True,9,"4,786,849","38,422",13,67,4.707,False
100000003016635100,0,True,9,"10,001,062","106,066",8,73,16.04,False
100000001530983100,1,False,9,"2,393,696","393,443",4,11,3.243,False
100000005933757100,2,False,9,"4,248,628","3,320",4,34,2.41,False
100000000437046100,2,False,9,"8,713,424","36,890",9,42,6.87,False
100000001857829100,2,False,9,"3,279,528","3,996",6,16,2.691,False
100000008165763100,1,False,9,"2,433,673","188,882",15,17,1.088,False


In [26]:
res = pd.DataFrame(findings)
display(res)
res.to_csv("checks_findings.csv", index=False, encoding="utf-8-sig")
reg.reset_index().to_csv("registry_nodes_enriched.csv", index=False, encoding="utf-8-sig")

,пункт,проверка,значение,вывод
0,1,доля транзитных (pass 0.8–1.2): топ-100 по in+...,2% vs 3%,сильного перекоса нет
1,1,"пересечение топ-100 по in+out и по max(in,out)",91,почти не меняется
2,2,не-seed узлов с out > in,335,"у них не виден часть входа, баланс считать нельзя"
3,2,"невидимый вход у них в сумме, KZT",229 908 004,= 62.8% оборота графа
4,2,из них первый исходящий раньше первого видимог...,186,"деньги были на счёте до выборки, т.е. вход из-..."
5,3,узлов без исходящих всего,1554,
6,3,из них обрыв 4-го колена,444,"не terminal, а «нет данных»"
7,3,"кандидаты в настоящие стоки (колено<4, не seed...",1079,сумма входа 136 739 480 KZT
8,3,"доля денег графа, ушедших в обрезанные узлы",15.5%,
9,3,обрезанных в топ-100 по входу,17,наивное правило «много пришло и не ушло» пойма...


In [27]:
import numpy as np, pandas as pd, networkx as nx
from IPython.display import display
pd.set_option("display.max_colwidth", 200)

need = ["seed_share", "seed_money_in", "in_cycle", "volume_dedup", "flag_truncated", "flag_out_gt_in", "pass_ratio"]
missing = [c for c in need if c not in reg.columns]
assert not missing, f"Сначала прогони ячейки проверок, не хватает колонок: {missing}"

F = reg.copy()
F.index.name = "gid"
seed_set = set(F.index[F.is_seed])
TX_END = tx.date.max()

# [A] сколько разных seed среди контрагентов (в обе стороны)
nb = pd.concat([e[["src", "dst"]].rename(columns={"src": "gid", "dst": "cp"}),
                e[["dst", "src"]].rename(columns={"dst": "gid", "src": "cp"})])
F["n_seed_links"] = nb[nb.cp.isin(seed_set)].groupby("gid").cp.nunique().reindex(F.index).fillna(0).astype(int)

# [C] доля исходящих денег, отправленных в течение 2 дней после какого-то входящего
out_t = tx[["src", "date", "sum_kzt"]].rename(columns={"src": "gid"}).sort_values("date")
in_t  = tx[["dst", "date"]].rename(columns={"dst": "gid", "date": "in_date"}).sort_values("in_date")
mm = pd.merge_asof(out_t, in_t, left_on="date", right_on="in_date", by="gid", direction="backward")
mm["fast"] = (mm.date - mm.in_date).dt.days.le(2)
F["fast_share"] = ((mm.sum_kzt * mm.fast).groupby(mm.gid).sum()
                   / mm.groupby("gid").sum_kzt.sum()).reindex(F.index)

# [A] максимум разных плательщиков за один день (синхронный сбор)
F["max_payers_day"] = (tx.groupby(["dst", tx.date.dt.normalize()]).src.nunique()
                       .groupby(level=0).max().reindex(F.index).fillna(0).astype(int))

# [A] последний входящий — в последние 3 дня месяца
F["late_last_in"] = (tx.groupby("dst").date.max().reindex(F.index) >= TX_END - pd.Timedelta(days=3)).fillna(False)

# [A] технические флаги
F["no_edges"] = (F.in_deg + F.out_deg) == 0
F["inflow_reliable"] = ~F.is_seed & ~F.flag_out_gt_in      # только тут pass_ratio похож на баланс
for c in ["in_deg", "out_deg", "in_from_seed_deg"]:
    F[c] = F[c].astype(int)

has_time = (tx.date != tx.date.dt.normalize()).any()
print("В датах есть время:", has_time,
      "" if has_time else "→ порядок переводов внутри дня неизвестен, fast_share — гипотеза (уровень C)")
print("Узлов с fast_share ≥ 0.7:", int((F.fast_share >= 0.7).sum()),
      "| с 3+ плательщиками в день:", int((F.max_payers_day >= 3).sum()),
      "| связаны с 2+ seed:", int((F.n_seed_links >= 2).sum()))

В датах есть время: False → порядок переводов внутри дня неизвестен, fast_share — гипотеза (уровень C)
Узлов с fast_share ≥ 0.7: 221 | с 3+ плательщиками в день: 38 | связаны с 2+ seed: 58


In [28]:
from sklearn.metrics import adjusted_rand_score

def consensus_clusters(G, n_runs=30, thr=0.8, seed0=0):
    U = nx.Graph(); U.add_nodes_from(G)
    for u, v, d in G.edges(data=True):
        if u == v: continue
        if U.has_edge(u, v): U[u][v]["w"] += d["sum_kzt"]
        else: U.add_edge(u, v, w=d["sum_kzt"])
    E = list(U.edges())
    together = np.zeros(len(E))
    for s in range(seed0, seed0 + n_runs):
        lab = {x: i for i, c in enumerate(nx.community.louvain_communities(U, weight="w", seed=s)) for x in c}
        together += np.fromiter((lab[u] == lab[v] for u, v in E), bool, len(E))
    agree = together / n_runs

    # ядра: связи, которые вместе в ≥ thr запусков
    C = nx.Graph(); C.add_nodes_from(U)
    C.add_edges_from(ed for ed, a in zip(E, agree) if a >= thr)
    core = {x: i for i, c in enumerate(nx.connected_components(C)) for x in c}
    csize = pd.Series(core).value_counts()
    ag = {}
    for (u, v), a in zip(E, agree):
        ag[(u, v)] = ag[(v, u)] = a
    # одиночки со связями присоединяем к соседу с максимальным согласием (если ≥ 0.5)
    for x in U.nodes():
        if csize[core[x]] == 1 and U.degree(x) > 0:
            best = max(U.neighbors(x), key=lambda y: ag[(x, y)])
            if ag[(x, best)] >= 0.5:
                core[x] = core[best]

    lab = pd.Series(core)
    iso = [x for x in U if U.degree(x) == 0]
    order = lab.drop(iso).value_counts().index
    lab = lab.map({c: i + 1 for i, c in enumerate(order)}).fillna(0).astype(int)  # 0 = нет переводов
    return lab, agree

cl_a, agree = consensus_clusters(G, seed0=0)
cl_b, _     = consensus_clusters(G, seed0=100)
F["cluster_id"] = cl_a.reindex(F.index).values

print("ARI между двумя независимыми консенсусами:",
      round(adjusted_rand_score(cl_a.reindex(F.index), cl_b.reindex(F.index)), 3), "(было 0.68–0.90 у одиночного Louvain)")
print("Доля связей с согласием ≥ 0.8:", f"{(agree >= 0.8).mean():.0%}")
sizes = F.cluster_id.value_counts()
print("Кластеров (без 0):", int((sizes.index != 0).sum()),
      "| размером ≥ 5:", int(((sizes >= 5) & (sizes.index != 0)).sum()),
      "| в кластере 0 (нет переводов):", int(sizes.get(0, 0)))
display(F.groupby("cluster_id").agg(n=("depth", "size"), seeds=("is_seed", "sum"),
                                     volume=("out_sum", "sum")).sort_values("n", ascending=False).head(15))

ARI между двумя независимыми консенсусами: 0.976 (было 0.68–0.90 у одиночного Louvain)
Доля связей с согласием ≥ 0.8: 81%
Кластеров (без 0): 87 | размером ≥ 5: 62 | в кластере 0 (нет переводов): 19


,n,seeds,volume
cluster_id,,,
1,270,1,"10,178,044"
2,261,1,"35,924,126"
3,140,3,"23,702,587"
4,125,7,"19,240,702"
5,123,0,"30,169,487"
6,119,0,"29,509,190"
7,102,0,"15,476,340"
8,61,0,"4,818,757"
9,56,1,"11,048,331"


In [30]:
ROLES = ["coordinator", "consolidator", "distributor", "transit", "terminal", "peripheral"]
ROLE_W = {"coordinator": 1.0, "consolidator": 0.9, "distributor": 0.8,
          "transit": 0.7, "terminal": 0.5, "peripheral": 0.1}

# Пороги: каждый должен объясняться одной фразой
P = dict(
    cons_min_in=5,        # консолидатор: от 5 разных плательщиков (в ТЗ сильные кандидаты — 8–24)
    cons_max_pass=0.5,    # ...и отдаёт дальше не больше половины
    dist_min_out=10,      # распределитель: от 10 получателей (в ТЗ сильные — 60–116)
    band=(0.8, 1.2),      # транзит: отдал 80–120% полученного (ориентир из ТЗ)
    min_sum=100_000,      # существенная сумма для транзита и терминала
    coord_min_in=5, coord_min_out=10,   # координатор: одновременно сборщик и раздающий
    fast_min=0.7,         # [C] ≥70% исходящих денег ушло в течение 2 дней после входа
    cons_min_seed_share=0.1,  # [C] ниже — связь с seed слабая
    term_min_seed_share=0.2,  # [C] терминал только при заметной доле seed-денег
)

def sat(x, lo, hi):   # плавная шкала 0..1 между порогом и «очень сильно»
    return np.clip((np.asarray(x, dtype=float) - lo) / (hi - lo), 0, 1)

def pct(x):           # перцентиль, у нулей = 0
    s = pd.Series(np.asarray(x, dtype=float))
    return ((s.rank(method="min") - 1) / (len(s) - 1)).values

def money(x):
    x = float(x)
    return f"{x/1e6:.1f} млн" if x >= 1e6 else f"{x/1e3:.0f} тыс" if x >= 1e3 else f"{x:.0f}"


def make_evidence(r, role, level):
    pre = "[seed] " if r.is_seed else ""
    reliable = r.inflow_reliable and r.in_sum > 0
    pr_txt = f"отдал дальше {r.pass_ratio:.0%}" if reliable else "видимый вход неполный"
    fast = 0 if pd.isna(r.fast_share) else r.fast_share

    if r.no_edges:
        txt = "seed без переводов в выгрузке: роль определить нельзя" if r.is_seed else "нет переводов в выгрузке"
    elif role == "coordinator":
        txt = f"признаки координации: {r.in_deg} плательщиков и {r.out_deg} получателей"
        if r.n_seed_links: txt += f", связан с {r.n_seed_links} seed"
        if r.in_cycle:     txt += ", есть круговые потоки"
    elif role == "consolidator":
        txt = f"признаки консолидации: {r.in_deg} плательщиков ({r.in_from_seed_deg} seed), получил {money(r.in_sum)}"
        txt += ", исходящие не выгружены" if r.flag_truncated else f", {pr_txt}"
        if level >= 2 and r.max_payers_day >= 3: txt += f", до {r.max_payers_day} плательщиков в день"
        if level == 3 and r.seed_share < P["cons_min_seed_share"] and r.n_seed_links == 0:
            txt += "; связь с seed слабая, возможен легальный хаб"
    elif role == "distributor":
        txt = f"веерная рассылка: {r.out_deg} получателей, отправил {money(r.out_sum)}"
        if r.flag_out_gt_in: txt += f", видимый вход {money(r.in_sum)}: источник вне выгрузки"
    elif role == "transit":
        txt = f"признаки транзита: получил {money(r.in_sum)}, отправил {money(r.out_sum)}"
        if reliable: txt += f" ({r.pass_ratio:.0%})"
        if level >= 2 and fast >= 0.5: txt += f", {fast:.0%} ушло в течение 2 дней"
    elif role == "terminal":
        txt = f"конечная точка в пределах выгрузки: получил {money(r.in_sum)} от {r.in_deg}, исходящих нет"
        if level == 3: txt += f", доля seed-денег {r.seed_share:.0%}"
        if level >= 2 and r.late_last_in: txt += "; вход в конце июля, мог переслать позже"
    else:
        if r.flag_truncated:
            txt = f"обрыв выгрузки на 4-м колене: получил {money(r.in_sum)} от {r.in_deg}, исходящие неизвестны"
        else:
            txt = f"выраженных признаков нет: {r.in_deg} вх./{r.out_deg} исх., оборот {money(r.volume_dedup)}"
    txt = pre + txt
    return txt if len(txt) <= 200 else txt[:199] + "…"


def assign_roles(F, level, p=P, with_evidence=True):
    g = lambda c: F[c].values
    n = len(F)
    pr = np.nan_to_num(g("pass_ratio").astype(float), nan=0.0)
    ind, outd, ins, outs = g("in_deg"), g("out_deg"), g("in_sum"), g("out_sum")
    trunc, seed, noe, reliable = g("flag_truncated"), g("is_seed"), g("no_edges"), g("inflow_reliable")
    fast = np.nan_to_num(g("fast_share").astype(float))

    # --- consolidator [B] ---
    gate_cons = (ind >= p["cons_min_in"]) & (pr <= p["cons_max_pass"])
    s_cons = (0.5 + 0.25 * sat(ind, p["cons_min_in"], 3 * p["cons_min_in"])
                  + 0.25 * (1 - np.clip(pr / p["cons_max_pass"], 0, 1)))
    if level == 1:
        gate_cons &= ~trunc                                  # у обрезанных «не отдаёт» — не факт
    else:
        s_cons = np.where(trunc, 0.7 * s_cons, s_cons)       # вход виден полностью, выход нет → штраф
        s_cons = s_cons + 0.1 * (g("max_payers_day") >= 3)   # синхронный сбор
    cons_weak = np.zeros(n, bool)
    if level == 3:                                           # [C] без связи с seed → возможен легальный хаб
        cons_weak = (g("seed_share") < p["cons_min_seed_share"]) & (g("n_seed_links") == 0)
        s_cons = np.where(cons_weak, 0.6 * s_cons, s_cons)

    # --- distributor [B] ---
    gate_dist = outd >= p["dist_min_out"]
    s_dist = 0.5 + 0.5 * sat(outd, p["dist_min_out"], 6 * p["dist_min_out"])

    # --- transit [B] / [C] по времени ---
    lo, hi = p["band"]
    in_band = (pr >= lo) & (pr <= hi) & (ins >= p["min_sum"])
    s_tr = (0.5 + 0.25 * (1 - np.clip(np.abs(pr - 1) / (hi - 1), 0, 1))
                + 0.25 * sat(ins, p["min_sum"], 10 * p["min_sum"]))
    tr_fast = np.zeros(n, bool)
    if level == 1:
        gate_tr = in_band
    else:   # у seed и out>in баланс не виден, поэтому смотрим на скорость
        tr_fast = ((fast >= p["fast_min"]) & (ins >= p["min_sum"]) & (outs >= p["min_sum"])
                   & ~(in_band & reliable))
        gate_tr = (in_band & reliable) | tr_fast
        s_tr = np.where(tr_fast, 0.5 + 0.5 * sat(fast, p["fast_min"], 1.0), s_tr)
        s_tr = s_tr + 0.1 * (in_band & (fast >= p["fast_min"]))

    # --- coordinator [B] / [C] по кругам ---
    gate_coord = (ind >= p["coord_min_in"]) & (outd >= p["coord_min_out"])
    s_coord = (0.5 + 0.25 * sat(ind, p["coord_min_in"], 3 * p["coord_min_in"])
                   + 0.25 * sat(outd, p["coord_min_out"], 6 * p["coord_min_out"]))
    coord_cyc = np.zeros(n, bool)
    if level == 3:
        coord_cyc = (ind >= 3) & (outd >= 3) & g("in_cycle") & (g("n_seed_links") >= 2) & ~gate_coord
        s_coord = np.where(coord_cyc, 0.5 + 0.25 * sat(g("n_seed_links"), 2, 6)
                                          + 0.25 * sat(g("seed_money_in"), 1e5, 1e6), s_coord)
        gate_coord = gate_coord | coord_cyc

    # --- terminal [B] / [C] ---
    gate_term = (outd == 0) & (g("depth") < 4) & ~seed & (ins >= p["min_sum"])   # обрезанные никогда
    s_term = 0.5 + 0.5 * sat(ins, p["min_sum"], 10 * p["min_sum"])
    if level >= 2:
        s_term = np.where(g("late_last_in"), 0.6 * s_term, s_term)
    if level == 3:
        gate_term &= g("seed_share") >= p["term_min_seed_share"]

    # --- приоритет правил: последнее перезаписывает ---
    role = np.full(n, "peripheral", dtype=object)
    score = np.zeros(n)
    for name, gate, s in [("terminal", gate_term, s_term), ("transit", gate_tr, s_tr),
                          ("distributor", gate_dist, s_dist), ("consolidator", gate_cons, s_cons),
                          ("coordinator", gate_coord, s_coord)]:
        role = np.where(gate, name, role)
        score = np.where(gate, s, score)

    # --- периферия и артефакты данных [A] ---
    per = role == "peripheral"
    small = (ind + outd <= 2) & (g("volume_dedup") < p["min_sum"])
    score = np.where(per, np.where(small, 0.9, 0.6), score)
    art_trunc = per & trunc
    score = np.where(art_trunc, 0.3, score)
    score = np.where(noe, 0.2, score)
    score = np.clip(score, 0, 1).round(3)

    tier = np.full(n, "B", dtype=object)
    tier = np.where(noe | art_trunc, "A", tier)
    hyp = ((tr_fast & (role == "transit")) | (coord_cyc & (role == "coordinator"))
           | ((role == "terminal") & (level == 3)) | (cons_weak & (role == "consolidator")))
    tier = np.where(hyp, "C", tier)

    # --- priority: по одной метрике от каждой семьи ---
    rw = np.array([ROLE_W[r] for r in role]) * score
    p_in, p_out, p_vol = pct(ind), pct(outd), pct(g("volume_dedup"))
    link = sat(g("n_seed_links"), 0, 3)
    if level == 1:
        prio = 0.4 * rw + 0.2 * p_in + 0.2 * p_out + 0.2 * p_vol
    elif level == 2:
        prio = 0.4 * rw + 0.15 * p_in + 0.15 * p_out + 0.15 * p_vol + 0.15 * link
    else:
        prio = (0.35 * rw + 0.2 * pct(g("seed_money_in")) + 0.1 * p_in + 0.1 * p_out
                + 0.1 * p_vol + 0.1 * link + 0.05 * g("in_cycle").astype(float))

    out = pd.DataFrame({"role": role, "role_score": score, "priority_score": np.round(prio, 4),
                        "tier": tier, "cluster_id": F.cluster_id.values}, index=F.index)
    if with_evidence:
        out["evidence"] = [make_evidence(r, rl, level) for r, rl in zip(F.itertuples(), role)]
    return out

In [31]:
def validate(df):
    checks = {
        "строк = 2248":        len(df) == 2248,
        "gid уникальны":       df.gid.is_unique,
        "роли из словаря":     df.role.isin(ROLES).all(),
        "role_score в [0,1]":  df.role_score.between(0, 1).all(),
        "priority в [0,1]":    df.priority_score.between(0, 1).all(),
        "нет пропусков":       df.notna().all().all(),
        "evidence непустой":   (df.evidence.str.len() > 0).all(),
        "evidence ≤ 200":      (df.evidence.str.len() <= 200).all(),
        "cluster_id целый":    pd.api.types.is_integer_dtype(df.cluster_id),
    }
    bad = [k for k, v in checks.items() if not v]
    return "OK" if not bad else f"ОШИБКИ: {bad}"

SCHEMA = ["gid", "role", "role_score", "cluster_id", "priority_score", "evidence"]
results = {}
for lvl, name in [(1, "v1_facts"), (2, "v2_corrected"), (3, "v3_hypotheses")]:
    res = assign_roles(F, lvl)
    results[name] = res
    strict = res.reset_index()[SCHEMA]
    strict["gid"] = strict["gid"].astype("int64")
    print(f"{name}: {validate(strict)}")
    strict.to_csv(f"nodes_roles_{name}.csv", index=False, encoding="utf-8-sig")        # строго по ТЗ
    res.reset_index().to_csv(f"nodes_roles_{name}_ext.csv", index=False, encoding="utf-8-sig")  # + tier

v1_facts: OK
v2_corrected: OK
v3_hypotheses: OK


In [32]:
names = list(results)

print("Число узлов по ролям:")
display(pd.DataFrame({k: v.role.value_counts() for k, v in results.items()}).reindex(ROLES).fillna(0).astype(int))

print("Уровень уверенности (A — факт, B — правило, C — гипотеза):")
display(pd.DataFrame({k: v.tier.value_counts() for k, v in results.items()}).fillna(0).astype(int))

print("v1 → v2 (строки v1, колонки v2):")
display(pd.crosstab(results["v1_facts"].role, results["v2_corrected"].role, margins=True))
print("v2 → v3:")
display(pd.crosstab(results["v2_corrected"].role, results["v3_hypotheses"].role, margins=True))

roles_all = pd.DataFrame({k: v.role for k, v in results.items()})
F["role_stable"] = roles_all.nunique(axis=1) == 1
print(f"Роль одинакова во всех трёх версиях: {F.role_stable.mean():.0%} узлов")
display(pd.crosstab(results["v2_corrected"].role, F.role_stable, margins=True)
          .rename(columns={True: "устойчиво", False: "зависит от версии"}))

print("Пересечение топ-20 по priority между версиями:")
tops20 = {k: set(v.nlargest(20, "priority_score").index) for k, v in results.items()}
display(pd.DataFrame([[len(tops20[a] & tops20[b]) for b in names] for a in names], index=names, columns=names))

for k, v in results.items():
    print(f"\nТоп-10 по priority, {k}:")
    display(v.nlargest(10, "priority_score")[["role", "role_score", "tier", "priority_score", "cluster_id", "evidence"]])

Число узлов по ролям:


,v1_facts,v2_corrected,v3_hypotheses
role,,,
coordinator,15,15,25
consolidator,25,25,25
distributor,49,49,43
transit,30,78,75
terminal,323,323,95
peripheral,1806,1758,1985


Уровень уверенности (A — факт, B — правило, C — гипотеза):


,v1_facts,v2_corrected,v3_hypotheses
tier,,,
A,463,463,463
B,1785,1722,1607
C,0,63,178


v1 → v2 (строки v1, колонки v2):


...

,role,role_score,tier,priority_score,cluster_id,evidence
gid,,,,,,
100000003684369100,coordinator,1,B,0.9991,3,"[seed] признаки координации: 24 плательщиков и 62 получателей, связан с 1 seed, есть круговые потоки"
100000008603629100,coordinator,1,B,0.9984,7,"признаки координации: 19 плательщиков и 61 получателей, есть круговые потоки"
100000006866783100,coordinator,0.95,B,0.9776,2,"[seed] признаки координации: 13 плательщиков и 67 получателей, есть круговые потоки"
100000003016635100,coordinator,0.825,B,0.928,4,"[seed] признаки координации: 8 плательщиков и 73 получателей, есть круговые потоки"
100000004400305100,coordinator,0.8,B,0.9173,2,"признаки координации: 7 плательщиков и 82 получателей, есть круговые потоки"
100000008477350100,coordinator,0.795,B,0.9147,2,"признаки координации: 11 плательщиков и 39 получателей, есть круговые потоки"
100000000437046100,coordinator,0.76,B,0.9019,2,"признаки координации: 9 плательщиков и 42 получателей, есть круговые потоки"
100000008165763100,coordinator,0.785,B,0.901,3,"признаки координации: 15 плательщиков и 17 получателей, связан с 1 seed"
100000002527114100,distributor,1,B,0.9001,1,"[seed] веерная рассылка: 111 получателей, отправил 3.3 млн"



Топ-10 по priority, v2_corrected:


,role,role_score,tier,priority_score,cluster_id,evidence
gid,,,,,,
100000003684369100,coordinator,1,B,0.8993,3,"[seed] признаки координации: 24 плательщиков и 62 получателей, связан с 1 seed, есть круговые потоки"
100000008603629100,coordinator,1,B,0.8488,7,"признаки координации: 19 плательщиков и 61 получателей, есть круговые потоки"
100000004015047100,consolidator,0.812,B,0.8354,13,"признаки консолидации: 9 плательщиков (3 seed), получил 919 тыс, отдал дальше 8%"
100000006866783100,coordinator,0.95,B,0.8282,2,"[seed] признаки координации: 13 плательщиков и 67 получателей, есть круговые потоки"
100000003115284100,consolidator,0.805,B,0.8103,15,"признаки консолидации: 8 плательщиков (2 seed), получил 2.2 млн, отдал дальше 24%, до 7 плательщиков в день"
100000008165763100,coordinator,0.785,B,0.8043,3,"признаки координации: 15 плательщиков и 17 получателей, связан с 1 seed"
100000008637537100,transit,1,C,0.798,43,"[seed] признаки транзита: получил 620 тыс, отправил 2.4 млн, 100% ушло в течение 2 дней"
100000005382566100,consolidator,0.736,B,0.7936,21,"[seed] признаки консолидации: 7 плательщиков (6 seed), получил 393 тыс, видимый вход неполный"
100000003016635100,coordinator,0.825,B,0.7785,4,"[seed] признаки координации: 8 плательщиков и 73 получателей, есть круговые потоки"



Топ-10 по priority, v3_hypotheses:


,role,role_score,tier,priority_score,cluster_id,evidence
gid,,,,,,
100000003684369100,coordinator,1,B,0.9309,3,"[seed] признаки координации: 24 плательщиков и 62 получателей, связан с 1 seed, есть круговые потоки"
100000004015047100,consolidator,0.812,B,0.8662,13,"признаки консолидации: 9 плательщиков (3 seed), получил 919 тыс, отдал дальше 8%"
100000008603629100,coordinator,1,B,0.8635,7,"признаки координации: 19 плательщиков и 61 получателей, есть круговые потоки"
100000003115284100,consolidator,0.805,B,0.8501,15,"признаки консолидации: 8 плательщиков (2 seed), получил 2.2 млн, отдал дальше 24%, до 7 плательщиков в день"
100000006866783100,coordinator,0.95,B,0.8495,2,"[seed] признаки координации: 13 плательщиков и 67 получателей, есть круговые потоки"
100000005382566100,consolidator,0.736,B,0.8303,21,"[seed] признаки консолидации: 7 плательщиков (6 seed), получил 393 тыс, видимый вход неполный"
100000003016635100,coordinator,0.825,B,0.8242,4,"[seed] признаки координации: 8 плательщиков и 73 получателей, есть круговые потоки"
100000008637537100,coordinator,0.643,C,0.8189,43,"[seed] признаки координации: 3 плательщиков и 3 получателей, связан с 2 seed, есть круговые потоки"
100000004070318100,coordinator,0.562,C,0.8097,2,"признаки координации: 4 плательщиков и 19 получателей, связан с 3 seed, есть круговые потоки"


In [33]:
from itertools import product

rows = []
for cmin, dmin, msum in product([3, 5, 8], [5, 10, 20], [50_000, 100_000, 300_000]):
    pp = {**P, "cons_min_in": cmin, "coord_min_in": cmin, "dist_min_out": dmin,
          "coord_min_out": dmin, "min_sum": msum}
    vc = assign_roles(F, 2, pp, with_evidence=False).role.value_counts()
    rows.append({"cons_min_in": cmin, "dist_min_out": dmin, "min_sum": msum, **vc.to_dict()})
sens = pd.DataFrame(rows).fillna(0)
display(sens[["cons_min_in", "dist_min_out", "min_sum"] + [r for r in ROLES if r in sens]])

,cons_min_in,dist_min_out,min_sum,coordinator,consolidator,distributor,transit,terminal,peripheral
0,3,5,50000,57,97,74,81,469,1470
1,3,5,100000,57,97,74,63,292,1665
2,3,5,300000,57,97,74,24,93,1903
3,3,10,50000,37,100,27,92,469,1523
4,3,10,100000,37,100,27,72,292,1720
5,3,10,300000,37,100,27,29,93,1962
6,3,20,50000,21,100,7,98,469,1553
7,3,20,100000,21,100,7,76,292,1752
8,3,20,300000,21,100,7,30,93,1997
9,5,5,50000,18,24,113,88,504,1501


In [34]:
import re
from itertools import product

P4 = dict(
    cons_min_in=5, cons_strong_in=8, cons_max_pass=0.5,   # консолидатор: 5+ плательщиков, выраженно от 8 (ориентир ТЗ 8–24)
    dist_min_out=10,                                      # распределитель: 10+ получателей
    band=(0.8, 1.2), min_sum=100_000,                     # транзит: пропуск 80–120% и существенная сумма
    coord_min_in=5, coord_min_out=10,                     # координатор: и собирает, и раздаёт...
    coord_strong_in=8, coord_min_seed_links=2,            # ...и (8+ плательщиков или связь с 2+ seed)
    fast_min=0.7,                                         # [C] 70%+ исходящих ушло в течение 2 дней
    weak_seed_share=0.1, term_seed_share=0.2,             # [C] пороги «слабой связи» с seed-деньгами
    seed_factor=0.85,                                     # seed уже известны → приоритет чуть ниже
)

def plural(n, one, few, many):
    n = abs(int(n))
    if n % 10 == 1 and n % 100 != 11: return one
    if 2 <= n % 10 <= 4 and not 12 <= n % 100 <= 14: return few
    return many

def cnt(n, one, few, many):
    return f"{int(n)} {plural(n, one, few, many)}"


def evidence_v4(r, role, weak):
    reliable = r.inflow_reliable and r.in_sum > 0
    pay = cnt(r.in_deg, "плательщик", "плательщика", "плательщиков")
    rec = cnt(r.out_deg, "получатель", "получателя", "получателей")
    fs = 0 if pd.isna(r.fast_share) else r.fast_share
    parts = []

    if r.no_edges:
        main = "seed без переводов в выгрузке: роль определить нельзя" if r.is_seed else "нет переводов в выгрузке"
    elif role == "coordinator":
        main = f"признаки координации: {pay} и {rec}"
    elif role == "consolidator":
        main = f"признаки консолидации: {pay}"
        if r.in_from_seed_deg: main += f" (из них seed: {r.in_from_seed_deg})"
        main += f", получил {money(r.in_sum)}"
        parts.append("исходящие не выгружены" if r.flag_truncated
                     else f"отдал дальше {r.pass_ratio:.0%}" if reliable else "видимый вход неполный")
        if r.max_payers_day >= 3: parts.append(f"до {r.max_payers_day} плательщиков в день")
    elif role == "distributor":
        main = f"веерная рассылка: {rec}, отправил {money(r.out_sum)}"
        if r.flag_out_gt_in: parts.append(f"видимый вход {money(r.in_sum)}, источник вне выгрузки")
    elif role == "transit":
        main = f"признаки транзита: получил {money(r.in_sum)}, отправил {money(r.out_sum)}"
        if reliable: main += f" ({r.pass_ratio:.0%})"
        if fs >= 0.5: parts.append(f"{fs:.0%} отправлено в течение 2 дней после входа")
    elif role == "terminal":
        main = (f"конечная точка в пределах выгрузки: получил {money(r.in_sum)} "
                f"от {cnt(r.in_deg, 'плательщика', 'плательщиков', 'плательщиков')}, исходящих нет")
        if r.late_last_in: parts.append("вход в конце июля, мог переслать позже")
    elif r.flag_truncated:
        main = (f"обрыв выгрузки на 4-м колене: получил {money(r.in_sum)} "
                f"от {cnt(r.in_deg, 'плательщика', 'плательщиков', 'плательщиков')}, исходящие неизвестны")
    else:
        main = f"выраженных признаков нет: {r.in_deg} вх./{r.out_deg} исх., оборот {money(r.volume_dedup)}"

    if not r.no_edges:
        if r.n_seed_links: parts.append(f"связан {'ещё ' if r.is_seed else ''}с {r.n_seed_links} seed")
        if r.in_cycle:     parts.append("участвует в круговых потоках")
        if role != "peripheral":
            if weak:                 parts.append("связь с seed-деньгами слабая, возможна легальная активность")
            elif r.seed_share >= 0.3: parts.append(f"доля seed-денег ~{r.seed_share:.0%}")

    text = ("[seed] " if r.is_seed else "") + main
    for part in parts:                       # добавляем по важности, пока влезает в 200 символов
        if len(text) + 2 + len(part) <= 200:
            text += "; " + part
    return text[:200]


def assign_roles_v4(F, p=P4, with_evidence=True):
    g = lambda c: F[c].values
    n = len(F)
    pr = np.nan_to_num(g("pass_ratio").astype(float), nan=0.0)
    ind, outd, ins, outs = g("in_deg"), g("out_deg"), g("in_sum"), g("out_sum")
    trunc, seed, noe, reliable = (g(c).astype(bool) for c in
                                  ["flag_truncated", "is_seed", "no_edges", "inflow_reliable"])
    fast = np.nan_to_num(g("fast_share").astype(float))
    links, sshare = g("n_seed_links"), g("seed_share").astype(float)
    late, cyc = g("late_last_in").astype(bool), g("in_cycle").astype(bool)

    weak = (sshare < p["weak_seed_share"]) & (links == 0)   # [C] влияет только на score
    hyp = np.where(weak, 0.75, 1.0)

    # coordinator [B]
    gate_coord = ((ind >= p["coord_min_in"]) & (outd >= p["coord_min_out"])
                  & ((ind >= p["coord_strong_in"]) | (links >= p["coord_min_seed_links"])))
    s_coord = (0.5 + 0.2 * sat(ind, p["coord_min_in"], 3 * p["coord_min_in"])
                   + 0.2 * sat(outd, p["coord_min_out"], 6 * p["coord_min_out"])
                   + 0.1 * sat(links, 0, 3)) * hyp

    # consolidator [B]: ступенька 5→8→16 плательщиков
    keep = np.where(trunc, 0.5, 1 - np.clip(pr / p["cons_max_pass"], 0, 1))  # у обрезанных «удержание» неизвестно
    gate_cons = (ind >= p["cons_min_in"]) & (pr <= p["cons_max_pass"])
    s_cons = (0.45 + 0.15 * sat(ind, p["cons_min_in"], p["cons_strong_in"])
                   + 0.15 * sat(ind, p["cons_strong_in"], 2 * p["cons_strong_in"])
                   + 0.15 * keep + 0.1 * (g("max_payers_day") >= 3))
    s_cons = np.where(trunc, 0.8 * s_cons, s_cons) * hyp

    # distributor [B]
    gate_dist = outd >= p["dist_min_out"]
    s_dist = (0.5 + 0.5 * sat(outd, p["dist_min_out"], 6 * p["dist_min_out"])) * hyp

    # transit: [B] по балансу, где он виден; [C] по скорости, где не виден
    lo, hi = p["band"]
    band_ok = reliable & (pr >= lo) & (pr <= hi) & (ins >= p["min_sum"])
    fast_ok = ~reliable & (fast >= p["fast_min"]) & (ins >= p["min_sum"]) & (outs >= p["min_sum"])
    gate_tr = band_ok | fast_ok
    s_tr = np.where(band_ok,
                    0.55 + 0.2 * (1 - np.clip(np.abs(pr - 1) / (hi - 1), 0, 1))
                         + 0.15 * sat(ins, p["min_sum"], 10 * p["min_sum"]) + 0.1 * (fast >= p["fast_min"]),
                    0.45 + 0.35 * sat(fast, p["fast_min"], 1.0)) * hyp

    # terminal [B]; гипотезы только снижают уверенность
    gate_term = (outd == 0) & (g("depth") < 4) & ~seed & (ins >= p["min_sum"])
    s_term = (0.5 + 0.3 * sat(ins, p["min_sum"], 10 * p["min_sum"])
                  + 0.2 * sat(sshare, p["term_seed_share"], 0.8))
    s_term = np.where(late, 0.7 * s_term, s_term)
    s_term = np.where(sshare < p["term_seed_share"], 0.6 * s_term, s_term)

    role = np.full(n, "peripheral", dtype=object)
    score = np.zeros(n)
    for name, gate, s in [("terminal", gate_term, s_term), ("transit", gate_tr, s_tr),
                          ("distributor", gate_dist, s_dist), ("consolidator", gate_cons, s_cons),
                          ("coordinator", gate_coord, s_coord)]:
        role = np.where(gate, name, role)
        score = np.where(gate, s, score)

    per = role == "peripheral"
    small = (ind + outd <= 2) & (g("volume_dedup") < p["min_sum"])
    score = np.where(per, np.where(small, 0.9, 0.6), score)
    art_trunc = per & trunc
    score = np.where(art_trunc, 0.3, score)
    score = np.where(noe, 0.2, score)
    score = np.clip(score, 0, 1).round(3)

    tier = np.full(n, "B", dtype=object)
    tier = np.where(noe | art_trunc, "A", tier)
    tier = np.where((role == "transit") & fast_ok & ~band_ok, "C", tier)

    rw = np.array([ROLE_W[r] for r in role]) * score
    prio = (0.35 * rw + 0.2 * pct(g("seed_money_in")) + 0.1 * pct(ind) + 0.1 * pct(outd)
            + 0.1 * pct(g("volume_dedup")) + 0.1 * sat(links, 0, 3) + 0.05 * cyc)
    prio = np.where(seed, prio * p["seed_factor"], prio)

    out = pd.DataFrame({"role": role, "role_score": score, "priority_score": np.round(prio, 4),
                        "tier": tier, "weak_seed_link": weak, "cluster_id": F.cluster_id.values},
                       index=F.index)
    if with_evidence:
        out["evidence"] = [evidence_v4(r, rl, w) for r, rl, w in zip(F.itertuples(), role, weak)]
    return out


res4 = assign_roles_v4(F)
results["v4_final"] = res4
print("Схема nodes_roles:", validate(res4.reset_index()[SCHEMA]))
display(pd.DataFrame({k: v.role.value_counts() for k, v in results.items()}).reindex(ROLES).fillna(0).astype(int))

Схема nodes_roles: OK


,v1_facts,v2_corrected,v3_hypotheses,v4_final
role,,,,
coordinator,15,15,25,9
consolidator,25,25,25,25
distributor,49,49,43,55
transit,30,78,75,49
terminal,323,323,95,323
peripheral,1806,1758,1985,1787


In [35]:
X = res4.join(F.drop(columns=["cluster_id"]))
warn = lambda cond: "⚠ " if cond else "  "

# 1. Логические противоречия: везде должен быть 0
contra = {
    "transit с видимым балансом вне 0.8–1.2":        ((X.role == "transit") & X.inflow_reliable & ~X.pass_ratio.between(0.8, 1.2)).sum(),
    "consolidator, отдаёт > 50%":                     ((X.role == "consolidator") & (X.pass_ratio.fillna(0) > 0.5)).sum(),
    "consolidator < 5 плательщиков":                  ((X.role == "consolidator") & (X.in_deg < 5)).sum(),
    "coordinator без 8+ плательщиков и без 2+ seed":  ((X.role == "coordinator") & (X.in_deg < 8) & (X.n_seed_links < 2)).sum(),
    "distributor < 10 получателей":                   ((X.role == "distributor") & (X.out_deg < 10)).sum(),
    "terminal с исходящими / seed / 4-е колено":      ((X.role == "terminal") & ((X.out_deg > 0) | X.is_seed | (X.depth >= 4))).sum(),
    "обрезанный узел = terminal или transit":         (X.flag_truncated & X.role.isin(["terminal", "transit"])).sum(),
    "нет переводов, но роль не peripheral":           (X.no_edges & (X.role != "peripheral")).sum(),
    "evidence содержит nan/inf":                      X.evidence.str.contains(r"\bnan\b|\binf\b", case=False).sum(),
    "метка [seed] не совпадает с is_seed":            (X.evidence.str.startswith("[seed]") != X.is_seed).sum(),
    "evidence длиннее 200":                           (X.evidence.str.len() > 200).sum(),
}
print("1) ПРОТИВОРЕЧИЯ ПРАВИЛАМ (должны быть нули)")
for k, v in contra.items():
    print(f"{warn(v > 0)}{k}: {v}")

# 2. Устойчивость роли к сдвигу порогов (±1 шаг)
grid = list(product([4, 5, 6], [8, 10, 12], [75_000, 100_000, 150_000]))
agree_cnt = np.zeros(len(F))
for cmin, dmin, msum in grid:
    pp = {**P4, "cons_min_in": cmin, "coord_min_in": cmin, "dist_min_out": dmin,
          "coord_min_out": dmin, "min_sum": msum}
    agree_cnt += assign_roles_v4(F, pp, with_evidence=False).role.values == res4.role.values
res4["role_stability"] = agree_cnt / len(grid)
X["role_stability"] = res4.role_stability
print(f"\n2) УСТОЙЧИВОСТЬ РОЛИ к порогам ({len(grid)} вариантов)")
display(res4.groupby("role").role_stability.describe()[["count", "mean", "min", "50%"]])
border = res4[(res4.role != "peripheral") & (res4.role_stability < 0.7)]
print(f"{warn(len(border) > 0)}Пограничных активных ролей (устойчивость < 70%): {len(border)}")

# 3. Поиск новых перекосов
print("\n3) ВОЗМОЖНЫЕ НОВЫЕ ПРОБЛЕМЫ")
top20, top50 = X.nlargest(20, "priority_score"), X.nlargest(50, "priority_score")
checks_new = [
    ("seed в топ-20", int(top20.is_seed.sum()), top20.is_seed.sum() > 10,
     "топ забит уже известными — проверить seed_factor"),
    ("обрезанных в топ-50", int(top50.flag_truncated.sum()), top50.flag_truncated.sum() > 5,
     "в топе узлы с неполными данными"),
    ("разных кластеров в топ-20", top20.cluster_id.nunique(), top20.cluster_id.nunique() <= 2,
     "топ из 1–2 кластеров — нет охвата сети"),
    ("кластер 0 (без переводов) в топ-50", int((top50.cluster_id == 0).sum()),
     (top50.cluster_id == 0).sum() > 0, "узлы без данных в топе"),
    ("активных ролей со слабой связью с seed", int((X.weak_seed_link & (X.role != "peripheral")).sum()),
     False, "кандидаты в легальную активность; в evidence это указано"),
    ("peripheral в топ-50 (звенья между seed)", int((top50.role == "peripheral").sum()),
     False, "роль слабая, но есть связи с seed/круги; проверить вручную"),
    ("совпадающих priority в топ-50", int(top50.priority_score.duplicated().sum()),
     top50.priority_score.duplicated().sum() > 0, "одинаковый скор → порядок случайный"),
    ("tier C (гипотеза) в топ-20", int((top20.tier == "C").sum()), (top20.tier == "C").sum() > 5,
     "верх топа держится на допущениях"),
]
for name, val, bad, hint in checks_new:
    print(f"{warn(bad)}{name}: {val}" + (f"  → {hint}" if bad else ""))

print("\nСредний priority по коленам (не должен быть монотонно «чем ближе к seed, тем выше» без причины):")
display(X.groupby("depth").priority_score.agg(["mean", "max", "size"]))

print("Как изменились роли v3 → v4:")
display(pd.crosstab(results["v3_hypotheses"].role, res4.role, margins=True))

print("Пограничные узлы (для ручной проверки):")
display(X.loc[border.index, ["role", "role_score", "role_stability", "in_deg", "out_deg", "pass_ratio",
                             "in_sum", "evidence"]].sort_values("role_stability").head(15))

1) ПРОТИВОРЕЧИЯ ПРАВИЛАМ (должны быть нули)
  transit с видимым балансом вне 0.8–1.2: 0
  consolidator, отдаёт > 50%: 0
  consolidator < 5 плательщиков: 0
  coordinator без 8+ плательщиков и без 2+ seed: 0
  distributor < 10 получателей: 0
  terminal с исходящими / seed / 4-е колено: 0
  обрезанный узел = terminal или transit: 0
  нет переводов, но роль не peripheral: 0
  evidence содержит nan/inf: 0
  метка [seed] не совпадает с is_seed: 0
  evidence длиннее 200: 0

2) УСТОЙЧИВОСТЬ РОЛИ к порогам (27 вариантов)


,count,mean,min,50%
role,,,,
consolidator,25,0.8933,0.6667,1
coordinator,9,1,1,1
distributor,55,0.9071,0.4444,1
peripheral,"1,787",0.9825,0.6667,1
terminal,323,0.9143,0.6667,1
transit,49,0.9252,0.6667,1


⚠ Пограничных активных ролей (устойчивость < 70%): 116

3) ВОЗМОЖНЫЕ НОВЫЕ ПРОБЛЕМЫ
  seed в топ-20: 1
  обрезанных в топ-50: 0
  разных кластеров в топ-20: 12
  кластер 0 (без переводов) в топ-50: 0
  активных ролей со слабой связью с seed: 278
  peripheral в топ-50 (звенья между seed): 0
  совпадающих priority в топ-50: 0
  tier C (гипотеза) в топ-20: 0

Средний priority по коленам (не должен быть монотонно «чем ближе к seed, тем выше» без причины):


,mean,max,size
depth,,,
0,0.2488,0.7714,81
1,0.3875,0.8454,472
2,0.3053,0.7563,462
3,0.2108,0.686,789
4,0.09907,0.267,444


Как изменились роли v3 → v4:


role,consolidator,coordinator,distributor,peripheral,terminal,transit,All
role,,,,,,,
consolidator,25,0,0,0,0,0,25
coordinator,0,9,12,2,0,2,25
distributor,0,0,43,0,0,0,43
peripheral,0,0,0,1757,228,0,1985
terminal,0,0,0,0,95,0,95
transit,0,0,0,28,0,47,75
All,25,9,55,1787,323,49,2248


Пограничные узлы (для ручной проверки):


,role,role_score,role_stability,in_deg,out_deg,pass_ratio,in_sum,evidence
gid,,,,,,,,
100000007639767100,distributor,0.5,0.4444,4,10,9.044,"90,913","веерная рассылка: 10 получателей, отправил 822 тыс; видимый вход 91 тыс, источник вне выгрузки; связан с 2 seed; участвует в круговых потоках"
100000001530983100,distributor,0.51,0.4444,4,11,3.243,"564,158","веерная рассылка: 11 получателей, отправил 1.8 млн; видимый вход 564 тыс, источник вне выгрузки; связан с 2 seed"
100000004007116100,transit,0.721,0.6667,3,4,0.1935,"764,704","[seed] признаки транзита: получил 765 тыс, отправил 148 тыс; 93% отправлено в течение 2 дней после входа; участвует в круговых потоках; доля seed-денег ~100%"
100000001686328100,terminal,0.216,0.6667,1,0,0,"140,000","конечная точка в пределах выгрузки: получил 140 тыс от 1 плательщика, исходящих нет; вход в конце июля, мог переслать позже; связь с seed-деньгами слабая, возможна легальная активность"
100000003073635100,terminal,0.306,0.6667,1,0,0,"129,231","конечная точка в пределах выгрузки: получил 129 тыс от 1 плательщика, исходящих нет; связь с seed-деньгами слабая, возможна легальная активность"
100000001303311100,consolidator,0.525,0.6667,5,0,0,"1,324,000","признаки консолидации: 5 плательщиков, получил 1.3 млн; отдал дальше 0%; до 4 плательщиков в день; связь с seed-деньгами слабая, возможна легальная активность"
100000000802149100,terminal,0.301,0.6667,1,0,0,"104,128","конечная точка в пределах выгрузки: получил 104 тыс от 1 плательщика, исходящих нет; связь с seed-деньгами слабая, возможна легальная активность"
100000004292885100,terminal,0.303,0.6667,1,0,0,"112,700","конечная точка в пределах выгрузки: получил 113 тыс от 1 плательщика, исходящих нет; связь с seed-деньгами слабая, возможна легальная активность"
100000008571243100,terminal,0.307,0.6667,1,0,0,"134,000","конечная точка в пределах выгрузки: получил 134 тыс от 1 плательщика, исходящих нет; связь с seed-деньгами слабая, возможна легальная активность"


In [36]:
N = res4.join(F.drop(columns=["cluster_id"]))
cl_of = N.cluster_id
ei = e.assign(c_src=e.src.map(cl_of), c_dst=e.dst.map(cl_of))
internal, cross = ei[ei.c_src == ei.c_dst], ei[ei.c_src != ei.c_dst]
sum_int  = internal.groupby("c_src").sum_kzt.sum()
seed_int = internal[internal.src.isin(seed_set)].groupby("c_src").sum_kzt.sum()
out_ext  = cross.groupby("c_src").sum_kzt.sum()
in_ext   = cross.groupby("c_dst").sum_kzt.sum()


def cluster_hypothesis(cid, grp, st):
    if cid == 0:
        return "Узлы без переводов в выгрузке (в т.ч. seed): структуру определить нельзя, нужны данные о входящих/других периодах"
    n, ns = st["n_nodes"], st["n_seed"]
    top = grp.sort_values("priority_score", ascending=False)
    first = lambda r: top.index[top.role == r][0] if (top.role == r).any() else None
    c, k, d, t = first("coordinator"), first("consolidator"), first("distributor"), first("transit")
    nodes_txt = cnt(n, "узел", "узла", "узлов")

    if n <= 3:
        main = (f"Малый фрагмент ({nodes_txt}) вокруг seed, связи за пределами выгрузки не видны" if ns
                else f"Малый фрагмент ({nodes_txt}) без выраженных ролей")
    elif c is not None and ns >= 2:
        main = f"Возможное ядро: координирующий узел {c} связан с группой из {ns} seed, признаки сбора и раздачи средств"
    elif c is not None:
        main = f"Узел {c} с признаками координации (собирает и раздаёт) в окружении {nodes_txt}"
    elif k is not None and (ns >= 1 or st["seed_share_internal"] >= 0.3):
        main = (f"Возможный сбор средств: точка консолидации {k} получает от "
                f"{cnt(grp.loc[k, 'in_deg'], 'плательщика', 'плательщиков', 'плательщиков')}, в т.ч. связанных с seed")
    elif d is not None:
        main = f"Веерная раздача: узел {d} распределяет средства на {cnt(grp.loc[d, 'out_deg'], 'получателя', 'получателей', 'получателей')}"
    elif t is not None:
        main = f"Возможная транзитная цепочка через узел {t}"
    elif ns:
        main = f"Окружение {ns} seed без выраженных ролей: преимущественно разовые получатели"
    else:
        main = "Периферийная группа без выраженных признаков"

    q = []
    if st["n_in_cycle"] >= 3:           q.append(f"{st['n_in_cycle']} в круговых потоках")
    if st["seed_share_internal"] >= 0.5: q.append("более половины внутреннего оборота исходит от seed")
    if st["n_truncated"] / n >= 0.5:     q.append("более половины узлов обрезаны 4-м коленом, картина неполная")
    lead = c or k or d
    if lead is not None and grp.loc[lead, "weak_seed_link"]:
        q.append("связь ключевого узла с seed-деньгами слабая")
    return "Гипотеза: " + main + ("; " + "; ".join(q) if q else "")


rows = []
for cid, grp in N.groupby("cluster_id"):
    top = grp.sort_values("priority_score", ascending=False)
    rc = grp.role.value_counts()
    si = float(sum_int.get(cid, 0))
    st = dict(
        cluster_id=int(cid), n_nodes=len(grp), n_seed=int(grp.is_seed.sum()),
        sum_kzt_internal=round(si, 2), top_gids=";".join(str(x) for x in top.index[:5]),
        sum_in_external=round(float(in_ext.get(cid, 0)), 2), sum_out_external=round(float(out_ext.get(cid, 0)), 2),
        seed_share_internal=round(float(seed_int.get(cid, 0)) / si, 3) if si else 0.0,
        n_in_cycle=int(grp.in_cycle.sum()), n_truncated=int(grp.flag_truncated.sum()),
        max_priority=float(top.priority_score.iloc[0]),
        **{f"n_{r}": int(rc.get(r, 0)) for r in ROLES},
    )
    st["hypothesis"] = cluster_hypothesis(cid, grp, st)
    rows.append(st)

clusters = pd.DataFrame(rows).sort_values("max_priority", ascending=False)
CL_SCHEMA = ["cluster_id", "n_nodes", "n_seed", "sum_kzt_internal", "top_gids", "hypothesis"]
clusters[CL_SCHEMA].sort_values("cluster_id").to_csv("clusters.csv", index=False, encoding="utf-8-sig")
clusters.sort_values("cluster_id").to_csv("clusters_ext.csv", index=False, encoding="utf-8-sig")

print(f"Кластеров: {len(clusters)} | ≥5 узлов: {(clusters.n_nodes >= 5).sum()} | "
      f"одиночек: {(clusters.n_nodes == 1).sum()} | с >1 seed: {(clusters.n_seed > 1).sum()}")
print(f"Сумма n_nodes = {clusters.n_nodes.sum()} (должно быть 2248), n_seed = {clusters.n_seed.sum()} (должно быть 81)")
print(f"Внутрикластерный оборот: {clusters.sum_kzt_internal.sum() / e.sum_kzt.sum():.0%} от всего графа")
display(clusters.head(15)[["cluster_id", "n_nodes", "n_seed", "sum_kzt_internal", "seed_share_internal",
                           "n_coordinator", "n_consolidator", "n_distributor", "n_in_cycle", "hypothesis"]])

Кластеров: 88 | ≥5 узлов: 63 | одиночек: 0 | с >1 seed: 10
Сумма n_nodes = 2248 (должно быть 2248), n_seed = 81 (должно быть 81)
Внутрикластерный оборот: 90% от всего графа


,cluster_id,n_nodes,n_seed,sum_kzt_internal,seed_share_internal,n_coordinator,n_consolidator,n_distributor,n_in_cycle,hypothesis
13,13,50,3,"5,695,865",0.404,0,1,1,13,"Гипотеза: Возможный сбор средств: точка консолидации 100000004015047100 получает от 9 плательщиков, в т.ч. связанных с seed; 13 в круговых потоках"
15,15,43,1,"5,318,439",0.634,1,1,0,9,Гипотеза: Узел 100000000343175100 с признаками координации (собирает и раздаёт) в окружении 43 узла; 9 в круговых потоках; более половины внутреннего оборота исходит от seed
3,3,140,3,"19,196,090",0.422,2,1,4,9,"Гипотеза: Возможное ядро: координирующий узел 100000008165763100 связан с группой из 3 seed, признаки сбора и раздачи средств; 9 в круговых потоках"
2,2,261,1,"28,765,253",0.11,3,8,8,36,Гипотеза: Узел 100000008477350100 с признаками координации (собирает и раздаёт) в окружении 261 узел; 36 в круговых потоках; связь ключевого узла с seed-деньгами слабая
19,19,30,1,"4,579,929",0.443,0,1,2,5,"Гипотеза: Возможный сбор средств: точка консолидации 100000002838861100 получает от 5 плательщиков, в т.ч. связанных с seed; 5 в круговых потоках"
12,12,50,3,"4,135,421",0,1,0,0,6,"Гипотеза: Возможное ядро: координирующий узел 100000008346837100 связан с группой из 3 seed, признаки сбора и раздачи средств; 6 в круговых потоках"
4,4,125,7,"18,300,397",0.508,1,1,2,21,"Гипотеза: Возможное ядро: координирующий узел 100000003016635100 связан с группой из 7 seed, признаки сбора и раздачи средств; 21 в круговых потоках; более половины внутреннего оборота исходит от ..."
7,7,102,0,"11,126,820",0,1,0,3,11,Гипотеза: Узел 100000008603629100 с признаками координации (собирает и раздаёт) в окружении 102 узла; 11 в круговых потоках; связь ключевого узла с seed-деньгами слабая
1,1,270,1,"10,178,044",0.323,0,0,3,10,Гипотеза: Веерная раздача: узел 100000001697501100 распределяет средства на 38 получателей; 10 в круговых потоках
60,60,5,1,"2,809,200",0.51,0,0,0,4,Гипотеза: Возможная транзитная цепочка через узел 100000004135268100; 4 в круговых потоках; более половины внутреннего оборота исходит от seed


In [37]:
TOP_N = 50   # ТЗ требует ≥ 20

p_seedmoney = pd.Series(pct(N.seed_money_in.values), index=N.index)
p_volume    = pd.Series(pct(N.volume_dedup.values), index=N.index)

def why(r):
    reasons = []
    if p_seedmoney[r.Index] >= 0.9: reasons.append(f"верхние 10% по seed-деньгам (~{money(r.seed_money_in)})")
    if r.n_seed_links >= 2:         reasons.append(f"прямые связи с {r.n_seed_links} seed")
    if r.in_cycle:                  reasons.append("участвует в круговых потоках")
    if r.max_payers_day >= 3:       reasons.append("синхронные поступления от нескольких плательщиков")
    if p_volume[r.Index] >= 0.95:   reasons.append(f"крупный оборот ({money(r.volume_dedup)})")
    if not reasons:                 reasons.append("совокупность структурных признаков")
    txt = (f"{r.evidence}. Приоритет: {', '.join(reasons)}. "
           f"Роль устойчива в {r.role_stability:.0%} вариантов порогов; кластер {r.cluster_id}.")
    if r.is_seed:    txt += " Уже известен (seed)."
    if r.tier == "C": txt += " Роль — гипотеза по косвенным признакам."
    return txt + " Требует проверки."

T = N.sort_values("priority_score", ascending=False).head(TOP_N).copy()
T.insert(0, "rank", range(1, len(T) + 1))
T["why"] = [why(r) for r in T.itertuples()]
T = T.reset_index()

TOP_SCHEMA = ["rank", "gid", "role", "priority_score", "why"]
T[TOP_SCHEMA].to_csv("top_nodes.csv", index=False, encoding="utf-8-sig")
T[TOP_SCHEMA + ["role_score", "tier", "role_stability", "is_seed", "depth", "cluster_id", "in_deg", "out_deg",
                "in_sum", "out_sum", "seed_money_in", "n_seed_links", "in_cycle"]] \
    .to_csv("top_nodes_ext.csv", index=False, encoding="utf-8-sig")

print("Состав топа по ролям:"); display(T.role.value_counts())
print(f"seed в топе: {T.is_seed.sum()} | кластеров: {T.cluster_id.nunique()} | tier C: {(T.tier == 'C').sum()}")
display(T.head(20)[["rank", "gid", "role", "priority_score", "why"]])

Состав топа по ролям:


role
distributor     16
transit         14
consolidator    11
coordinator      9
Name: count, dtype: int64

seed в топе: 8 | кластеров: 23 | tier C: 10


,rank,gid,role,priority_score,why
0,1,100000004015047100,consolidator,0.8454,"признаки консолидации: 9 плательщиков (из них seed: 3), получил 919 тыс; отдал дальше 8%; связан с 3 seed; участвует в круговых потоках; доля seed-денег ~64%. Приоритет: верхние 10% по seed-деньга..."
1,2,100000003115284100,consolidator,0.8416,"признаки консолидации: 8 плательщиков (из них seed: 2), получил 2.2 млн; отдал дальше 24%; до 7 плательщиков в день; связан с 2 seed; участвует в круговых потоках; доля seed-денег ~39%. Приоритет:..."
2,3,100000008165763100,coordinator,0.7849,"признаки координации: 15 плательщиков и 17 получателей; связан с 1 seed. Приоритет: верхние 10% по seed-деньгам (~189 тыс), синхронные поступления от нескольких плательщиков, крупный оборот (1.3 м..."
3,4,100000004070318100,distributor,0.7782,"веерная рассылка: 19 получателей, отправил 552 тыс; видимый вход 207 тыс, источник вне выгрузки; связан с 3 seed; участвует в круговых потоках. Приоритет: прямые связи с 3 seed, участвует в кругов..."
4,5,100000004299488100,consolidator,0.7769,"признаки консолидации: 6 плательщиков (из них seed: 1), получил 1.7 млн; отдал дальше 20%; до 3 плательщиков в день; связан с 1 seed; участвует в круговых потоках. Приоритет: верхние 10% по seed-д..."
5,6,100000003684369100,coordinator,0.7714,"[seed] признаки координации: 24 плательщика и 62 получателя; связан ещё с 1 seed; участвует в круговых потоках; доля seed-денег ~100%. Приоритет: верхние 10% по seed-деньгам (~500 тыс), участвует ..."
6,7,100000002838861100,consolidator,0.7574,"признаки консолидации: 5 плательщиков (из них seed: 2), получил 555 тыс; отдал дальше 2%; связан с 2 seed; участвует в круговых потоках; доля seed-денег ~73%. Приоритет: верхние 10% по seed-деньга..."
7,8,100000008686313100,consolidator,0.7571,"признаки консолидации: 6 плательщиков (из них seed: 1), получил 1.9 млн; отдал дальше 17%; связан с 1 seed; участвует в круговых потоках. Приоритет: верхние 10% по seed-деньгам (~233 тыс), участву..."
8,9,100000008346837100,coordinator,0.7563,"признаки координации: 9 плательщиков и 25 получателей; связан с 1 seed; участвует в круговых потоках. Приоритет: участвует в круговых потоках, синхронные поступления от нескольких плательщиков. Ро..."
9,10,100000007055802100,consolidator,0.751,"признаки консолидации: 6 плательщиков (из них seed: 1), получил 2.0 млн; отдал дальше 30%; до 4 плательщиков в день; связан с 1 seed; участвует в круговых потоках. Приоритет: верхние 10% по seed-д..."


In [38]:
nodes_out = res4.reset_index()[SCHEMA]
nodes_out["gid"] = nodes_out.gid.astype("int64")
nodes_out.to_csv("nodes_roles.csv", index=False, encoding="utf-8-sig")
res4.reset_index().to_csv("nodes_roles_ext.csv", index=False, encoding="utf-8-sig")

nr = pd.read_csv("nodes_roles.csv")
cl = pd.read_csv("clusters.csv")
tp = pd.read_csv("top_nodes.csv")

final = {
    "nodes_roles: схема ТЗ":                  validate(nr) == "OK" and list(nr.columns) == SCHEMA,
    "clusters: схема ТЗ":                     list(cl.columns) == CL_SCHEMA,
    "top_nodes: схема ТЗ":                    list(tp.columns) == TOP_SCHEMA,
    "каждый cluster_id узла есть в clusters": set(nr.cluster_id) == set(cl.cluster_id),
    "сумма n_nodes = 2248":                   cl.n_nodes.sum() == 2248,
    "сумма n_seed = 81":                      cl.n_seed.sum() == 81,
    "n_nodes совпадает с nodes_roles":        (nr.cluster_id.value_counts().sort_index()
                                               == cl.set_index("cluster_id").n_nodes.sort_index()).all(),
    "top_gids принадлежат своему кластеру":   all(nr.set_index("gid").loc[[int(x) for x in s.split(";")], "cluster_id"].eq(c).all()
                                                  for c, s in zip(cl.cluster_id, cl.top_gids)),
    "hypothesis заполнена":                   cl.hypothesis.str.len().gt(0).all(),
    "top_nodes ≥ 20 строк":                   len(tp) >= 20,
    "rank = 1..N":                            tp["rank"].tolist() == list(range(1, len(tp) + 1)),
    "priority в топе по убыванию":            tp.priority_score.is_monotonic_decreasing,
    "gid и роли топа совпадают с nodes_roles": (tp.merge(nr, on="gid", suffixes=("", "_n"))
                                                .pipe(lambda d: (d.role == d.role_n).all() and len(d) == len(tp))),
    "why заполнен":                           tp.why.str.len().gt(0).all(),
}
for k, v in final.items():
    print(("✅ " if v else "❌ ") + k)

✅ nodes_roles: схема ТЗ
✅ clusters: схема ТЗ
✅ top_nodes: схема ТЗ
✅ каждый cluster_id узла есть в clusters
✅ сумма n_nodes = 2248
✅ сумма n_seed = 81
✅ n_nodes совпадает с nodes_roles
✅ top_gids принадлежат своему кластеру
✅ hypothesis заполнена
✅ top_nodes ≥ 20 строк
✅ rank = 1..N
✅ priority в топе по убыванию
✅ gid и роли топа совпадают с nodes_roles
✅ why заполнен


In [39]:
# 1) Разделяем взаимные пары и настоящие круги
F["in_cycle3"] = F.index.isin({n for c in cycles if len(c) >= 3 for n in c})
mut = edge_reg[edge_reg.mutual]
F["n_mutual"] = mut.groupby("src").dst.nunique().reindex(F.index).fillna(0).astype(int)
print("в кругах любой длины:", int(F.in_cycle.sum()),
      "| в кругах из 3+ участников:", int(F.in_cycle3.sum()),
      "| есть взаимные пары:", int((F.n_mutual > 0).sum()))

# 4) Направление денег у распределителей: платят seed или получают от seed?
dist = res4.index[res4.role == "distributor"]
chk = pd.DataFrame({
    "платит seed (кол-во)":    e[e.src.isin(dist) & e.dst.isin(seed_set)].groupby("src").dst.nunique(),
    "платит seed (сумма)":     e[e.src.isin(dist) & e.dst.isin(seed_set)].groupby("src").sum_kzt.sum(),
    "получает от seed (кол-во)": e[e.dst.isin(dist) & e.src.isin(seed_set)].groupby("dst").src.nunique(),
}).fillna(0)
chk = chk[chk.sum(axis=1) > 0].join(res4.priority_score).sort_values("priority_score", ascending=False)
display(chk.head(20))

в кругах любой длины: 300 | в кругах из 3+ участников: 121 | есть взаимные пары: 266


,платит seed (кол-во),платит seed (сумма),получает от seed (кол-во),priority_score
100000004070318100,3,"126,813",1,0.7782
100000001697501100,1,"90,000",1,0.7475
100000008304139100,2,"598,000",1,0.7451
100000004156082100,0,0,1,0.7418
100000004351795100,1,"39,800",1,0.7331
100000008748914100,1,"1,582,700",1,0.7275
100000004847758100,1,"56,000",1,0.7032
100000007639767100,2,"20,000",1,0.699
100000001530983100,0,0,2,0.6968
100000005910114100,2,"318,389",0,0.686
